In [ ]:
%load_ext autoreload
%autoreload 2

### Sweep over lambda values per assay and characterize behavior

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name not in ("CLSBN26", "PepCL_Manuscript"):    
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find CLSBN26 project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "Data"
SRC_DIR = PROJECT_ROOT / "src"
MODEL_CHECKPOINTS_DIR = PROJECT_ROOT / "Model_Checkpoints"
FIGURES_DIR = PROJECT_ROOT / "Figures"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("FIGURES_DIR:", FIGURES_DIR)
print("PROJECT_ROOT in sys.path:", str(PROJECT_ROOT) in sys.path)

PROJECT_ROOT: /home/pchati/PepCL_Manuscript
DATA_DIR: /home/pchati/PepCL_Manuscript/Data
FIGURES_DIR: /home/pchati/PepCL_Manuscript/Figures
PROJECT_ROOT in sys.path: True


In [ ]:
# imports
from src.general_utils import *
from src.get_feature_data import *
from src.Model.mhcprime import *
from src.model_utils import *
from src.general_utils import *
from src.get_feature_data import *
from src.pepcl import *
from src.mhcprime_inference import *
from src.plotting import *
from src.pepcl_logging import *
from src.lambda_variations_utils import *

# misc
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# change gpu
# GPU_ID = 0
# GPU_ID = 1
# GPU_ID = 2
GPU_ID = 3

torch.cuda.set_device(GPU_ID)
print(f"Using GPU: {torch.cuda.get_device_name(GPU_ID)} (ID: {torch.cuda.current_device()})")
# device="cuda"
device = torch.device(f"cuda:{GPU_ID}")

Using GPU: NVIDIA H100 80GB HBM3 (ID: 3)


### Figure output directory

In [ ]:
# create figures and data dir
lambda_var_figures_dir=f"{FIGURES_DIR}/Revisions/Lambda_Variation_Figures"
lambda_var_figures_data_dir=f"{lambda_var_figures_dir}/Associated_Data"

os.makedirs(lambda_var_figures_dir, exist_ok=True)
os.makedirs(lambda_var_figures_data_dir, exist_ok=True)

### Load and process results

#### Format pseudosequences and get feature data

In [ ]:
# format pseudosequences
from Data.mhc_pseudo_seqs import mhc_seq_dict
mhc_seq_dict={k.removeprefix("HLA-").replace("*","").replace(":",""):v for k,v in mhc_seq_dict.items()}

In [ ]:
# get feature table and processed features
aa_features_table_ext, descriptor_groups=load_feature_data()
processed_feature_table, corr_info = process_aa_feature_table(aa_features_table_ext)
feature_names = list(processed_feature_table.columns)  # pass to FeatureProjectionEncoder

#### Load final validation performances only first

 - Will need to use run-specific sets in their individual directories.

#### YD experiments

In [ ]:
%%time
# load multiple dfs
path_prefix="YD"

results_dirs=[
    f"{DATA_DIR}/Lambda_Variations_and_Component_Ablations/{path_prefix}_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_1_Seed/", 
    f"{DATA_DIR}/Lambda_Variations_and_Component_Ablations/{path_prefix}_Lambda_Ablations_w_Alt_MHCPrime_DR_Only_Default_Params_1_Seed/",
    f"{DATA_DIR}/Lambda_Variations_and_Component_Ablations/{path_prefix}_Lambda_Ablations_w_Alt_MHCPrime_Remaining_4_Seeds/",
]

df_names=["ms_val", "holec_val"]


loaded_dfs_all=load_and_merge_dir_dfs(results_dirs, df_names=df_names, key_cols=("seq", "allele", "label"), how="outer")

ms_val_for_holec=loaded_dfs_all["ms_val"]
holec_val=loaded_dfs_all["holec_val"]

# match the ms val to the assay val
ms_val_for_holec_matched, holec_val=filter_by_allele_len_pairs(ms_val_for_holec, holec_val, allele_col="allele", seq_col="seq", filter_both=False)

# create dfs dict
holec_dfs={
    "ms_val_matched": ms_val_for_holec_matched,
    "assay_val": holec_val,
}

for df_name, df in holec_dfs.items():
    print(df_name)
    print_data_stats(df)
    print(end="\n")

[/home/pchati/PepCL_Manuscript/Data/Lambda_Variations_and_Component_Ablations/YD_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_1_Seed/] found 2 df(s) (filtered to 2 requested)


dir 0: YD_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_1_Seed:   0%|          | 0/2 [00:00<?, ?it/s]

dir 0: YD_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_1_Seed: 100%|██████████| 2/2 [00:00<00:00,  6.70it/s]


[/home/pchati/PepCL_Manuscript/Data/Lambda_Variations_and_Component_Ablations/YD_Lambda_Ablations_w_Alt_MHCPrime_DR_Only_Default_Params_1_Seed/] found 2 df(s) (filtered to 2 requested)


dir 1: YD_Lambda_Ablations_w_Alt_MHCPrime_DR_Only_Default_Params_1_Seed: 100%|██████████| 2/2 [00:00<00:00,  4.30it/s]


[/home/pchati/PepCL_Manuscript/Data/Lambda_Variations_and_Component_Ablations/YD_Lambda_Ablations_w_Alt_MHCPrime_Remaining_4_Seeds/] found 2 df(s) (filtered to 2 requested)


dir 2: YD_Lambda_Ablations_w_Alt_MHCPrime_Remaining_4_Seeds: 100%|██████████| 2/2 [00:01<00:00,  1.99it/s]


total unique dfs in store: 2
ms_val_matched
Total: 80,000
Pos: 800 | 1.00%
Neg: 79,200 | 99.00%
N alleles: 16

assay_val
Total: 72,300
Pos: 723 | 1.00%
Neg: 71,577 | 99.00%
N alleles: 16

CPU times: user 1.74 s, sys: 358 ms, total: 2.1 s
Wall time: 2.09 s


In [ ]:
# get datasplits
datasplits=[1, 2, 3, 4]

path_prefix="YD"

results_dirs=[
    f"{DATA_DIR}/Lambda_Variations_and_Component_Ablations/{path_prefix}_Lambda_Ablations_w_Alt_MHCPrime_CV_Datasplit_{ds}/" for ds in datasplits
]

df_names=["ms_val", "holec_val"]

holec_dfs_ds={}

for ds, rd in zip(tqdm(datasplits), results_dirs):
    seed_num=ds+4 # offset by 4
    ms_val=pd.read_csv(f"{rd}/ms_val.csv")
    holec_val=pd.read_csv(f"{rd}/holec_val.csv")
    ms_val.columns=ms_val.columns.str.replace("seed_0", f"seed_{seed_num}")
    holec_val.columns=holec_val.columns.str.replace("seed_0", f"seed_{seed_num}")
    holec_dfs_ds[f"ms_val_matched_{ds}"]=ms_val
    holec_dfs_ds[f"assay_val_{ds}"]=holec_val

  0%|          | 0/4 [00:00<?, ?it/s]

100%|██████████| 4/4 [00:01<00:00,  2.14it/s]


In [ ]:
# save for lambda cross over test
# save_dict_pickle(holec_dfs, f"{DATA_DIR}/Revisions_Data/lambda_crossover_holec_dfs.pkl")

#### EpiScan experiments

 - Forgot to keep `dataset_source` in R9/RT, we can just add back after (simple fix). For future runs, the script has already been corrected.

##### R9

In [ ]:
%%time
# load multiple dfs
path_prefix="EpiScan_R9"

results_dirs=[
    f"{DATA_DIR}/Lambda_Variations_and_Component_Ablations/{path_prefix}_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_1_Seed/", 
    f"{DATA_DIR}/Lambda_Variations_and_Component_Ablations/{path_prefix}_Lambda_Ablations_w_Alt_MHCPrime_DR_Only_Default_Params_1_Seed/",
    f"{DATA_DIR}/Lambda_Variations_and_Component_Ablations/{path_prefix}_Lambda_Ablations_w_Alt_MHCPrime_Remaining_4_Seeds/",
]


df_names=["ms_val", "episcan_r9_rt_val"]

loaded_dfs_all=load_and_merge_dir_dfs(results_dirs, df_names=df_names, key_cols=("seq", "allele", "label"), how="outer")

ms_val_for_episcan_r9=loaded_dfs_all["ms_val"]
episcan_r9_rt_val_for_r9=loaded_dfs_all["episcan_r9_rt_val"]

# match the ms val to the assay val
ms_val_for_episcan_r9_matched, episcan_r9_rt_val_for_r9=filter_by_allele_len_pairs(ms_val_for_episcan_r9, episcan_r9_rt_val_for_r9, allele_col="allele", seq_col="seq", filter_both=False)

# create dfs dict
episcan_r9_dfs={
    "ms_val_matched": ms_val_for_episcan_r9_matched,
    "assay_val": episcan_r9_rt_val_for_r9,
    "r9_val": episcan_r9_rt_val_for_r9.query("dataset_source == 'r9'"),
    "rt_val": episcan_r9_rt_val_for_r9.query("dataset_source == 'rt'"),
}

for df_name, df in episcan_r9_dfs.items():
    print(df_name)
    print_data_stats(df)
    print(end="\n")

[/home/pchati/PepCL_Manuscript/Data/Lambda_Variations_and_Component_Ablations/EpiScan_R9_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_1_Seed/] found 2 df(s) (filtered to 2 requested)


dir 0: EpiScan_R9_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_1_Seed:   0%|          | 0/2 [00:00<?, ?it/s]

dir 0: EpiScan_R9_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_1_Seed: 100%|██████████| 2/2 [00:00<00:00, 13.25it/s]


[/home/pchati/PepCL_Manuscript/Data/Lambda_Variations_and_Component_Ablations/EpiScan_R9_Lambda_Ablations_w_Alt_MHCPrime_DR_Only_Default_Params_1_Seed/] found 2 df(s) (filtered to 2 requested)


dir 1: EpiScan_R9_Lambda_Ablations_w_Alt_MHCPrime_DR_Only_Default_Params_1_Seed: 100%|██████████| 2/2 [00:00<00:00,  8.00it/s]


[/home/pchati/PepCL_Manuscript/Data/Lambda_Variations_and_Component_Ablations/EpiScan_R9_Lambda_Ablations_w_Alt_MHCPrime_Remaining_4_Seeds/] found 2 df(s) (filtered to 2 requested)


dir 2: EpiScan_R9_Lambda_Ablations_w_Alt_MHCPrime_Remaining_4_Seeds: 100%|██████████| 2/2 [00:00<00:00,  3.71it/s]


total unique dfs in store: 2
ms_val_matched
Total: 20,000
Pos: 200 | 1.00%
Neg: 19,800 | 99.00%
N alleles: 4

assay_val
Total: 33,800
Pos: 338 | 1.00%
Neg: 33,462 | 99.00%
N alleles: 4

r9_val
Total: 13,800
Pos: 138 | 1.00%
Neg: 13,662 | 99.00%
N alleles: 4

rt_val
Total: 20,000
Pos: 200 | 1.00%
Neg: 19,800 | 99.00%
N alleles: 4

CPU times: user 1.14 s, sys: 89.7 ms, total: 1.23 s
Wall time: 1.22 s


In [ ]:
# get datasplits
datasplits=[1, 2, 3, 4]

path_prefix="EpiScan_R9"

results_dirs=[
    f"{DATA_DIR}/Lambda_Variations_and_Component_Ablations/{path_prefix}_Lambda_Ablations_w_Alt_MHCPrime_CV_Datasplit_{ds}/" for ds in datasplits
]

df_names=["ms_val", "episcan_r9_rt_val"]

episcan_r9_dfs_ds={}

for ds, rd in zip(tqdm(datasplits), results_dirs):
    seed_num=ds+4 # offset by 4
    ms_val=pd.read_csv(f"{rd}/ms_val.csv")
    episcan_r9_rt_val=pd.read_csv(f"{rd}/episcan_r9_rt_val.csv")
    ms_val.columns=ms_val.columns.str.replace("seed_0", f"seed_{seed_num}")
    episcan_r9_rt_val.columns=episcan_r9_rt_val.columns.str.replace("seed_0", f"seed_{seed_num}")
    episcan_r9_dfs_ds[f"ms_val_matched_{ds}"]=ms_val
    episcan_r9_dfs_ds[f"assay_val_{ds}"]=episcan_r9_rt_val

100%|██████████| 4/4 [00:00<00:00,  6.52it/s]


##### R9/RT

In [ ]:
%%time
# load multiple dfs
path_prefix="EpiScan_R9_RT"


results_dirs=[
    f"{DATA_DIR}/Lambda_Variations_and_Component_Ablations/{path_prefix}_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_1_Seed/", 
    f"{DATA_DIR}/Lambda_Variations_and_Component_Ablations/{path_prefix}_Lambda_Ablations_w_Alt_MHCPrime_DR_Only_Default_Params_1_Seed/",
    f"{DATA_DIR}/Lambda_Variations_and_Component_Ablations/{path_prefix}_Lambda_Ablations_w_Alt_MHCPrime_Remaining_4_Seeds/",
]

df_names=["ms_val", "episcan_r9_rt_val"]

loaded_dfs_all=load_and_merge_dir_dfs(results_dirs, df_names=df_names, key_cols=("seq", "allele", "label"), how="outer")

ms_val_for_episcan_r9rt=loaded_dfs_all["ms_val"]
episcan_r9_rt_val_for_r9rt=loaded_dfs_all["episcan_r9_rt_val"]

# match the ms val to the assay val
ms_val_for_episcan_r9rt_matched, episcan_r9_rt_val_for_r9rt=filter_by_allele_len_pairs(ms_val_for_episcan_r9rt, episcan_r9_rt_val_for_r9rt, allele_col="allele", seq_col="seq", filter_both=False)

# create dfs dict
episcan_r9rt_dfs={
    "ms_val_matched": ms_val_for_episcan_r9rt_matched,
    "assay_val": episcan_r9_rt_val_for_r9rt,
    "r9_val": episcan_r9_rt_val_for_r9rt.query("dataset_source == 'r9'"),
    "rt_val": episcan_r9_rt_val_for_r9rt.query("dataset_source == 'rt'"),
}

for df_name, df in episcan_r9rt_dfs.items():
    print(df_name)
    print_data_stats(df)
    print(end="\n")

[/home/pchati/PepCL_Manuscript/Data/Lambda_Variations_and_Component_Ablations/EpiScan_R9_RT_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_1_Seed/] found 2 df(s) (filtered to 2 requested)


dir 0: EpiScan_R9_RT_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_1_Seed: 100%|██████████| 2/2 [00:00<00:00, 20.49it/s]


[/home/pchati/PepCL_Manuscript/Data/Lambda_Variations_and_Component_Ablations/EpiScan_R9_RT_Lambda_Ablations_w_Alt_MHCPrime_DR_Only_Default_Params_1_Seed/] found 2 df(s) (filtered to 2 requested)


dir 1: EpiScan_R9_RT_Lambda_Ablations_w_Alt_MHCPrime_DR_Only_Default_Params_1_Seed: 100%|██████████| 2/2 [00:00<00:00, 11.88it/s]


[/home/pchati/PepCL_Manuscript/Data/Lambda_Variations_and_Component_Ablations/EpiScan_R9_RT_Lambda_Ablations_w_Alt_MHCPrime_Remaining_4_Seeds/] found 2 df(s) (filtered to 2 requested)


dir 2: EpiScan_R9_RT_Lambda_Ablations_w_Alt_MHCPrime_Remaining_4_Seeds: 100%|██████████| 2/2 [00:00<00:00,  5.76it/s]


total unique dfs in store: 2
ms_val_matched
Total: 20,000
Pos: 200 | 1.00%
Neg: 19,800 | 99.00%
N alleles: 4

assay_val
Total: 33,800
Pos: 338 | 1.00%
Neg: 33,462 | 99.00%
N alleles: 4

r9_val
Total: 13,800
Pos: 138 | 1.00%
Neg: 13,662 | 99.00%
N alleles: 4

rt_val
Total: 20,000
Pos: 200 | 1.00%
Neg: 19,800 | 99.00%
N alleles: 4

CPU times: user 853 ms, sys: 126 ms, total: 979 ms
Wall time: 970 ms


In [ ]:
# get datasplits
datasplits=[1, 2, 3, 4]

path_prefix="EpiScan_R9_RT"

results_dirs=[
    f"{DATA_DIR}/Lambda_Variations_and_Component_Ablations/{path_prefix}_Lambda_Ablations_w_Alt_MHCPrime_CV_Datasplit_{ds}/" for ds in datasplits
]


df_names=["ms_val", "episcan_r9_rt_val"]

episcan_r9rt_dfs_ds={}

for ds, rd in zip(tqdm(datasplits), results_dirs):
    seed_num=ds+4 # offset by 4
    ms_val=pd.read_csv(f"{rd}/ms_val.csv")
    episcan_r9_rt_val=pd.read_csv(f"{rd}/episcan_r9_rt_val.csv")
    ms_val.columns=ms_val.columns.str.replace("seed_0", f"seed_{seed_num}")
    episcan_r9_rt_val.columns=episcan_r9_rt_val.columns.str.replace("seed_0", f"seed_{seed_num}")
    episcan_r9rt_dfs_ds[f"ms_val_matched_{ds}"]=ms_val
    episcan_r9rt_dfs_ds[f"assay_val_{ds}"]=episcan_r9_rt_val

  0%|          | 0/4 [00:00<?, ?it/s]

100%|██████████| 4/4 [00:00<00:00,  8.92it/s]


#### BA experiments

In [ ]:
%%time
# load multiple dfs
path_prefix="BA_CON"


results_dirs=[
    f"{DATA_DIR}/Lambda_Variations_and_Component_Ablations/{path_prefix}_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_1_Seed/", 
    f"{DATA_DIR}/Lambda_Variations_and_Component_Ablations/{path_prefix}_Lambda_Ablations_w_Alt_MHCPrime_DR_Only_Default_Params_1_Seed/",
    f"{DATA_DIR}/Lambda_Variations_and_Component_Ablations/{path_prefix}_Lambda_Ablations_w_Alt_MHCPrime_Remaining_4_Seeds/",
]

df_names=["ms_val", "ba_val"]

loaded_dfs_all=load_and_merge_dir_dfs(results_dirs, df_names=df_names, key_cols=("seq", "allele", "label"), how="outer")

ms_val_for_ba_con=loaded_dfs_all["ms_val"]
ba_con_val=loaded_dfs_all["ba_val"]

# match the ms val to the assay val
ms_val_for_ba_con_matched, ba_con_val=filter_by_allele_len_pairs(ms_val_for_ba_con, ba_con_val, allele_col="allele", seq_col="seq", filter_both=False)

# create dfs dict
ba_con_dfs={
    "ms_val_matched": ms_val_for_ba_con_matched,
    "assay_val": ba_con_val,
}

for df_name, df in ba_con_dfs.items():
    print(df_name)
    print_data_stats(df)
    print(end="\n")

[/home/pchati/PepCL_Manuscript/Data/Lambda_Variations_and_Component_Ablations/BA_CON_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_1_Seed/] found 2 df(s) (filtered to 2 requested)


dir 0: BA_CON_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_1_Seed: 100%|██████████| 2/2 [00:00<00:00,  2.14it/s]


[/home/pchati/PepCL_Manuscript/Data/Lambda_Variations_and_Component_Ablations/BA_CON_Lambda_Ablations_w_Alt_MHCPrime_DR_Only_Default_Params_1_Seed/] found 2 df(s) (filtered to 2 requested)


dir 1: BA_CON_Lambda_Ablations_w_Alt_MHCPrime_DR_Only_Default_Params_1_Seed: 100%|██████████| 2/2 [00:01<00:00,  1.15it/s]


[/home/pchati/PepCL_Manuscript/Data/Lambda_Variations_and_Component_Ablations/BA_CON_Lambda_Ablations_w_Alt_MHCPrime_Remaining_4_Seeds/] found 2 df(s) (filtered to 2 requested)


dir 2: BA_CON_Lambda_Ablations_w_Alt_MHCPrime_Remaining_4_Seeds: 100%|██████████| 2/2 [00:03<00:00,  1.66s/it]


total unique dfs in store: 2
ms_val_matched
Total: 327,500
Pos: 3,275 | 1.00%
Neg: 324,225 | 99.00%
N alleles: 41

assay_val
Total: 3,052
Pos: 1,526 | 50.00%
Neg: 1,526 | 50.00%
N alleles: 41

CPU times: user 6.14 s, sys: 720 ms, total: 6.86 s
Wall time: 6.84 s


In [ ]:
# get datasplits
datasplits=[1, 2, 3, 4]

path_prefix="BA_CON"

results_dirs=[
    f"{DATA_DIR}/Lambda_Variations_and_Component_Ablations/{path_prefix}_Lambda_Ablations_w_Alt_MHCPrime_CV_Datasplit_{ds}/" for ds in datasplits
]


df_names=["ms_val", "ba_val"]

ba_con_dfs_ds={}

for ds, rd in zip(tqdm(datasplits), results_dirs):
    seed_num=ds+4 # offset by 4
    ms_val=pd.read_csv(f"{rd}/ms_val.csv")
    ba_con_val=pd.read_csv(f"{rd}/ba_val.csv")

    ms_val, ba_con_val=filter_by_allele_len_pairs(ms_val, ba_con_val, allele_col="allele", seq_col="seq", filter_both=False)

    ms_val.columns=ms_val.columns.str.replace("seed_0", f"seed_{seed_num}")
    ba_con_val.columns=ba_con_val.columns.str.replace("seed_0", f"seed_{seed_num}")
    ba_con_dfs_ds[f"ms_val_matched_{ds}"]=ms_val
    ba_con_dfs_ds[f"assay_val_{ds}"]=ba_con_val

  0%|          | 0/4 [00:00<?, ?it/s]

100%|██████████| 4/4 [00:04<00:00,  1.14s/it]


#### ESCAPE-seq experiments

In [ ]:
%%time
# load multiple dfs
path_prefix="ESCAPE_seq_CON"


results_dirs=[
    f"{DATA_DIR}/Lambda_Variations_and_Component_Ablations/{path_prefix}_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_Full_5_Seeds_Cleaned/",
    f"{DATA_DIR}/Lambda_Variations_and_Component_Ablations/{path_prefix}_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_Full_5_Seeds_Cleaned_FT_DR_Only/",

]

df_names=["ms_val", "escape_val"]

loaded_dfs_all=load_and_merge_dir_dfs(results_dirs, df_names=df_names, key_cols=("seq", "allele", "label"), how="outer")

ms_val_for_escape_con=loaded_dfs_all["ms_val"]
escape_con_val=loaded_dfs_all["escape_val"]

# match the ms val to the assay val
ms_val_for_escape_con_matched, escape_con_val=filter_by_allele_len_pairs(ms_val_for_escape_con, escape_con_val, allele_col="allele", seq_col="seq", filter_both=False)

# create dfs dict
escape_con_dfs={
    # "ms_val": ms_val_for_escape_con,
    "ms_val_matched": ms_val_for_escape_con_matched,
    "assay_val": escape_con_val,
}

for df_name, df in escape_con_dfs.items():
    print(df_name)
    print_data_stats(df)
    print(end="\n")

[/home/pchati/PepCL_Manuscript/Data/Lambda_Variations_and_Component_Ablations/ESCAPE_seq_CON_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_Full_5_Seeds_Cleaned/] found 2 df(s) (filtered to 2 requested)


dir 0: ESCAPE_seq_CON_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_Full_5_Seeds_Cleaned:   0%|          | 0/2 [00:00<?, ?it/s]

dir 0: ESCAPE_seq_CON_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_Full_5_Seeds_Cleaned: 100%|██████████| 2/2 [00:00<00:00,  2.77it/s]


[/home/pchati/PepCL_Manuscript/Data/Lambda_Variations_and_Component_Ablations/ESCAPE_seq_CON_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_Full_5_Seeds_Cleaned_FT_DR_Only/] found 2 df(s) (filtered to 2 requested)


dir 1: ESCAPE_seq_CON_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_Full_5_Seeds_Cleaned_FT_DR_Only: 100%|██████████| 2/2 [00:00<00:00,  3.00it/s]


total unique dfs in store: 2
ms_val_matched
Total: 185,000
Pos: 1,850 | 1.00%
Neg: 183,150 | 99.00%
N alleles: 37

assay_val
Total: 1,315
Pos: 657 | 49.96%
Neg: 658 | 50.04%
N alleles: 41

CPU times: user 1.67 s, sys: 182 ms, total: 1.85 s
Wall time: 1.84 s


In [ ]:
# get datasplits

def merge_keep_first_dups(df1, df2, key_cols=("seq","allele","label"), how="outer"):
    key_cols = list(key_cols)
    dup_cols = [c for c in df2.columns if c in df1.columns and c not in key_cols]
    df2_trimmed = df2.drop(columns=dup_cols)
    return df1.merge(df2_trimmed, on=key_cols, how=how)

datasplits=[1, 2, 3, 4]

path_prefix="ESCAPE_seq_CON"

results_dirs=[
    f"{DATA_DIR}/Lambda_Variations_and_Component_Ablations/{path_prefix}_Lambda_Ablations_w_Alt_MHCPrime_Cleaned_CV_Datasplit_{ds}/" for ds in datasplits
]

results_dirs_2=[f"{DATA_DIR}/Lambda_Variations_and_Component_Ablations/{path_prefix}_Lambda_Ablations_w_Alt_MHCPrime_Cleaned_FT_DR_Only_CV_Datasplit_{ds}/" for ds in datasplits]

df_names=["ms_val", "escape_val"]

escape_con_dfs_ds={}

for ds in tqdm(datasplits):

    rd1=f"{DATA_DIR}/Lambda_Variations_and_Component_Ablations/{path_prefix}_Lambda_Ablations_w_Alt_MHCPrime_Cleaned_CV_Datasplit_{ds}/"
    rd2=f"{DATA_DIR}/Lambda_Variations_and_Component_Ablations/{path_prefix}_Lambda_Ablations_w_Alt_MHCPrime_Cleaned_FT_DR_Only_CV_Datasplit_{ds}/"

    seed_num=ds+4 # offset by 4
    ms_val_1=pd.read_csv(f"{rd1}/ms_val.csv")
    ms_val_2=pd.read_csv(f"{rd2}/ms_val.csv")

    ms_val=merge_keep_first_dups(ms_val_1, ms_val_2, key_cols=("seq","allele","label"), how="outer")

    escape_con_val_1=pd.read_csv(f"{rd1}/escape_val.csv")
    escape_con_val_2=pd.read_csv(f"{rd2}/escape_val.csv")

    escape_con_val=merge_keep_first_dups(escape_con_val_1, escape_con_val_2, key_cols=("seq","allele","label"), how="outer")

    ms_val, escape_con_val=filter_by_allele_len_pairs(ms_val, escape_con_val, allele_col="allele", seq_col="seq", filter_both=False)

    ms_val.columns=ms_val.columns.str.replace("seed_0", f"seed_{seed_num}")
    escape_con_val.columns=escape_con_val.columns.str.replace("seed_0", f"seed_{seed_num}")
    escape_con_dfs_ds[f"ms_val_matched_{ds}"]=ms_val
    escape_con_dfs_ds[f"assay_val_{ds}"]=escape_con_val

  0%|          | 0/4 [00:00<?, ?it/s]

100%|██████████| 4/4 [00:04<00:00,  1.10s/it]


### Allele-aware sampling off ablations

#### YD experiments

In [ ]:
%%time
# load multiple dfs
path_prefix="YD"

results_dirs=[
    f"{DATA_DIR}/Lambda_Variations_and_Component_Ablations/{path_prefix}_Allele_Aware_Ablations_w_Alt_MHCPrime_Full_5_Seeds/", 
]

df_names=["ms_val", "holec_val"]


loaded_dfs_all=load_and_merge_dir_dfs(results_dirs, df_names=df_names, key_cols=("seq", "allele", "label"), how="outer")

ms_val_for_holec=loaded_dfs_all["ms_val"]
holec_val=loaded_dfs_all["holec_val"]

# match the ms val to the assay val
ms_val_for_holec_matched, holec_val=filter_by_allele_len_pairs(ms_val_for_holec, holec_val, allele_col="allele", seq_col="seq", filter_both=False)

# create dfs dict
holec_dfs_aa_off={
    "ms_val_matched": ms_val_for_holec_matched,
    "assay_val": holec_val,
}

for df_name, df in holec_dfs_aa_off.items():
    print(df_name)
    print_data_stats(df)
    print(end="\n")

[/home/pchati/PepCL_Manuscript/Data/Lambda_Variations_and_Component_Ablations/YD_Allele_Aware_Ablations_w_Alt_MHCPrime_Full_5_Seeds/] found 2 df(s) (filtered to 2 requested)


dir 0: YD_Allele_Aware_Ablations_w_Alt_MHCPrime_Full_5_Seeds:   0%|          | 0/2 [00:00<?, ?it/s]

dir 0: YD_Allele_Aware_Ablations_w_Alt_MHCPrime_Full_5_Seeds: 100%|██████████| 2/2 [00:00<00:00,  5.49it/s]


total unique dfs in store: 2
ms_val_matched
Total: 80,000
Pos: 800 | 1.00%
Neg: 79,200 | 99.00%
N alleles: 16

assay_val
Total: 72,300
Pos: 723 | 1.00%
Neg: 71,577 | 99.00%
N alleles: 16

CPU times: user 536 ms, sys: 31.6 ms, total: 567 ms
Wall time: 562 ms


In [ ]:
# get datasplits
datasplits=[1, 2, 3, 4]

path_prefix="YD"

results_dirs=[
    f"{DATA_DIR}/Lambda_Variations_and_Component_Ablations/{path_prefix}_Allele_Aware_Ablations_w_Alt_MHCPrime_CV_Datasplit_{ds}/" for ds in datasplits
]

df_names=["ms_val", "holec_val"]

holec_dfs_aa_off_ds={}

for ds, rd in zip(tqdm(datasplits), results_dirs):
    seed_num=ds+4 # offset by 4
    ms_val=pd.read_csv(f"{rd}/ms_val.csv")
    holec_val=pd.read_csv(f"{rd}/holec_val.csv")
    ms_val.columns=ms_val.columns.str.replace("seed_0", f"seed_{seed_num}")
    holec_val.columns=holec_val.columns.str.replace("seed_0", f"seed_{seed_num}")
    holec_dfs_aa_off_ds[f"ms_val_matched_{ds}"]=ms_val
    holec_dfs_aa_off_ds[f"assay_val_{ds}"]=holec_val

  0%|          | 0/4 [00:00<?, ?it/s]

100%|██████████| 4/4 [00:00<00:00,  4.51it/s]


#### EpiScan experiments

 - Forgot to keep `dataset_source` in R9/RT, we can just add back after (simple fix). For future runs, the script has already been corrected.

##### R9

In [ ]:
%%time
# load multiple dfs
path_prefix="EpiScan_R9"

results_dirs=[
    f"{DATA_DIR}/Lambda_Variations_and_Component_Ablations/{path_prefix}_Allele_Aware_Ablations_w_Alt_MHCPrime_Full_5_Seeds/", 
]


df_names=["ms_val", "episcan_r9_rt_val"]

loaded_dfs_all=load_and_merge_dir_dfs(results_dirs, df_names=df_names, key_cols=("seq", "allele", "label"), how="outer")

ms_val_for_episcan_r9=loaded_dfs_all["ms_val"]
episcan_r9_rt_val_for_r9=loaded_dfs_all["episcan_r9_rt_val"]

# match the ms val to the assay val
ms_val_for_episcan_r9_matched, episcan_r9_rt_val_for_r9=filter_by_allele_len_pairs(ms_val_for_episcan_r9, episcan_r9_rt_val_for_r9, allele_col="allele", seq_col="seq", filter_both=False)

# create dfs dict
episcan_r9_dfs_aa_off={
    "ms_val_matched": ms_val_for_episcan_r9_matched,
    "assay_val": episcan_r9_rt_val_for_r9,
    "r9_val": episcan_r9_rt_val_for_r9.query("dataset_source == 'r9'"),
    "rt_val": episcan_r9_rt_val_for_r9.query("dataset_source == 'rt'"),
}

for df_name, df in episcan_r9_dfs_aa_off.items():
    print(df_name)
    print_data_stats(df)
    print(end="\n")

[/home/pchati/PepCL_Manuscript/Data/Lambda_Variations_and_Component_Ablations/EpiScan_R9_Allele_Aware_Ablations_w_Alt_MHCPrime_Full_5_Seeds/] found 2 df(s) (filtered to 2 requested)


dir 0: EpiScan_R9_Allele_Aware_Ablations_w_Alt_MHCPrime_Full_5_Seeds:   0%|          | 0/2 [00:00<?, ?it/s]

dir 0: EpiScan_R9_Allele_Aware_Ablations_w_Alt_MHCPrime_Full_5_Seeds: 100%|██████████| 2/2 [00:00<00:00, 13.24it/s]


total unique dfs in store: 2
ms_val_matched
Total: 20,000
Pos: 200 | 1.00%
Neg: 19,800 | 99.00%
N alleles: 4

assay_val
Total: 33,800
Pos: 338 | 1.00%
Neg: 33,462 | 99.00%
N alleles: 4

r9_val
Total: 13,800
Pos: 138 | 1.00%
Neg: 13,662 | 99.00%
N alleles: 4

rt_val
Total: 20,000
Pos: 200 | 1.00%
Neg: 19,800 | 99.00%
N alleles: 4

CPU times: user 255 ms, sys: 27.8 ms, total: 283 ms
Wall time: 280 ms


In [ ]:
# get datasplits
datasplits=[1, 2, 3, 4]

path_prefix="EpiScan_R9"

results_dirs=[
    f"{DATA_DIR}/Lambda_Variations_and_Component_Ablations/{path_prefix}_Allele_Aware_Ablations_w_Alt_MHCPrime_CV_Datasplit_{ds}/" for ds in datasplits
]

df_names=["ms_val", "episcan_r9_rt_val"]

episcan_r9_dfs_aa_off_ds={}

for ds, rd in zip(tqdm(datasplits), results_dirs):
    seed_num=ds+4 # offset by 4
    ms_val=pd.read_csv(f"{rd}/ms_val.csv")
    episcan_r9_rt_val=pd.read_csv(f"{rd}/episcan_r9_rt_val.csv")
    ms_val.columns=ms_val.columns.str.replace("seed_0", f"seed_{seed_num}")
    episcan_r9_rt_val.columns=episcan_r9_rt_val.columns.str.replace("seed_0", f"seed_{seed_num}")
    episcan_r9_dfs_aa_off_ds[f"ms_val_matched_{ds}"]=ms_val
    episcan_r9_dfs_aa_off_ds[f"assay_val_{ds}"]=episcan_r9_rt_val

  0%|          | 0/4 [00:00<?, ?it/s]

100%|██████████| 4/4 [00:00<00:00, 11.75it/s]


##### R9/RT

In [ ]:
%%time
# load multiple dfs
path_prefix="EpiScan_R9_RT"


results_dirs=[
    f"{DATA_DIR}/Lambda_Variations_and_Component_Ablations/{path_prefix}_Allele_Aware_Ablations_w_Alt_MHCPrime_Full_5_Seeds/", 
]

df_names=["ms_val", "episcan_r9_rt_val"]

loaded_dfs_all=load_and_merge_dir_dfs(results_dirs, df_names=df_names, key_cols=("seq", "allele", "label"), how="outer")

ms_val_for_episcan_r9rt=loaded_dfs_all["ms_val"]
episcan_r9_rt_val_for_r9rt=loaded_dfs_all["episcan_r9_rt_val"]

# match the ms val to the assay val
ms_val_for_episcan_r9rt_matched, episcan_r9_rt_val_for_r9rt=filter_by_allele_len_pairs(ms_val_for_episcan_r9rt, episcan_r9_rt_val_for_r9rt, allele_col="allele", seq_col="seq", filter_both=False)

# create dfs dict
episcan_r9rt_dfs_aa_off={
    "ms_val_matched": ms_val_for_episcan_r9rt_matched,
    "assay_val": episcan_r9_rt_val_for_r9rt,
    "r9_val": episcan_r9_rt_val_for_r9rt.query("dataset_source == 'r9'"),
    "rt_val": episcan_r9_rt_val_for_r9rt.query("dataset_source == 'rt'"),
}

for df_name, df in episcan_r9rt_dfs_aa_off.items():
    print(df_name)
    print_data_stats(df)
    print(end="\n")

[/home/pchati/PepCL_Manuscript/Data/Lambda_Variations_and_Component_Ablations/EpiScan_R9_RT_Allele_Aware_Ablations_w_Alt_MHCPrime_Full_5_Seeds/] found 2 df(s) (filtered to 2 requested)


dir 0: EpiScan_R9_RT_Allele_Aware_Ablations_w_Alt_MHCPrime_Full_5_Seeds: 100%|██████████| 2/2 [00:00<00:00, 13.95it/s]


total unique dfs in store: 2
ms_val_matched
Total: 20,000
Pos: 200 | 1.00%
Neg: 19,800 | 99.00%
N alleles: 4

assay_val
Total: 33,800
Pos: 338 | 1.00%
Neg: 33,462 | 99.00%
N alleles: 4

r9_val
Total: 13,800
Pos: 138 | 1.00%
Neg: 13,662 | 99.00%
N alleles: 4

rt_val
Total: 20,000
Pos: 200 | 1.00%
Neg: 19,800 | 99.00%
N alleles: 4

CPU times: user 242 ms, sys: 19.3 ms, total: 261 ms
Wall time: 258 ms


In [ ]:
# get datasplits
datasplits=[1, 2, 3, 4]

path_prefix="EpiScan_R9_RT"

results_dirs=[
    f"{DATA_DIR}/Lambda_Variations_and_Component_Ablations/{path_prefix}_Allele_Aware_Ablations_w_Alt_MHCPrime_CV_Datasplit_{ds}/" for ds in datasplits
]


df_names=["ms_val", "episcan_r9_rt_val"]

episcan_r9rt_dfs_aa_off_ds={}

for ds, rd in zip(tqdm(datasplits), results_dirs):
    seed_num=ds+4 # offset by 4
    ms_val=pd.read_csv(f"{rd}/ms_val.csv")
    episcan_r9_rt_val=pd.read_csv(f"{rd}/episcan_r9_rt_val.csv")
    ms_val.columns=ms_val.columns.str.replace("seed_0", f"seed_{seed_num}")
    episcan_r9_rt_val.columns=episcan_r9_rt_val.columns.str.replace("seed_0", f"seed_{seed_num}")
    episcan_r9rt_dfs_aa_off_ds[f"ms_val_matched_{ds}"]=ms_val
    episcan_r9rt_dfs_aa_off_ds[f"assay_val_{ds}"]=episcan_r9_rt_val

  0%|          | 0/4 [00:00<?, ?it/s]

100%|██████████| 4/4 [00:00<00:00, 11.67it/s]


#### BA experiments

In [ ]:
%%time
# load multiple dfs
path_prefix="BA_CON"


results_dirs=[
    f"{DATA_DIR}/Lambda_Variations_and_Component_Ablations/{path_prefix}_Allele_Aware_Ablations_w_Alt_MHCPrime_Full_5_Seeds/", 
]

df_names=["ms_val", "ba_val"]

loaded_dfs_all=load_and_merge_dir_dfs(results_dirs, df_names=df_names, key_cols=("seq", "allele", "label"), how="outer")

ms_val_for_ba_con=loaded_dfs_all["ms_val"]
ba_con_val=loaded_dfs_all["ba_val"]

# match the ms val to the assay val
ms_val_for_ba_con_matched, ba_con_val=filter_by_allele_len_pairs(ms_val_for_ba_con, ba_con_val, allele_col="allele", seq_col="seq", filter_both=False)

# create dfs dict
ba_con_dfs_aa_off={
    "ms_val_matched": ms_val_for_ba_con_matched,
    "assay_val": ba_con_val,
}

for df_name, df in ba_con_dfs_aa_off.items():
    print(df_name)
    print_data_stats(df)
    print(end="\n")

[/home/pchati/PepCL_Manuscript/Data/Lambda_Variations_and_Component_Ablations/BA_CON_Allele_Aware_Ablations_w_Alt_MHCPrime_Full_5_Seeds/] found 2 df(s) (filtered to 2 requested)


dir 0: BA_CON_Allele_Aware_Ablations_w_Alt_MHCPrime_Full_5_Seeds:   0%|          | 0/2 [00:00<?, ?it/s]

dir 0: BA_CON_Allele_Aware_Ablations_w_Alt_MHCPrime_Full_5_Seeds: 100%|██████████| 2/2 [00:00<00:00,  2.26it/s]


total unique dfs in store: 2
ms_val_matched
Total: 327,500
Pos: 3,275 | 1.00%
Neg: 324,225 | 99.00%
N alleles: 41

assay_val
Total: 3,052
Pos: 1,526 | 50.00%
Neg: 1,526 | 50.00%
N alleles: 41

CPU times: user 1.25 s, sys: 190 ms, total: 1.44 s
Wall time: 1.43 s


In [ ]:
# get datasplits
datasplits=[1, 2, 3, 4]

path_prefix="BA_CON"

results_dirs=[
    f"{DATA_DIR}/Lambda_Variations_and_Component_Ablations/{path_prefix}_Allele_Aware_Ablations_w_Alt_MHCPrime_CV_Datasplit_{ds}/" for ds in datasplits
]


df_names=["ms_val", "ba_val"]

ba_con_dfs_aa_off_ds={}

for ds, rd in zip(tqdm(datasplits), results_dirs):
    seed_num=ds+4 # offset by 4
    ms_val=pd.read_csv(f"{rd}/ms_val.csv")
    ba_con_val=pd.read_csv(f"{rd}/ba_val.csv")

    ms_val, ba_con_val=filter_by_allele_len_pairs(ms_val, ba_con_val, allele_col="allele", seq_col="seq", filter_both=False)

    ms_val.columns=ms_val.columns.str.replace("seed_0", f"seed_{seed_num}")
    ba_con_val.columns=ba_con_val.columns.str.replace("seed_0", f"seed_{seed_num}")
    ba_con_dfs_aa_off_ds[f"ms_val_matched_{ds}"]=ms_val
    ba_con_dfs_aa_off_ds[f"assay_val_{ds}"]=ba_con_val

  0%|          | 0/4 [00:00<?, ?it/s]

100%|██████████| 4/4 [00:03<00:00,  1.11it/s]


#### ESCAPE-seq experiments

In [ ]:
%%time
# load multiple dfs
path_prefix="ESCAPE_seq_CON"


results_dirs=[
    f"{DATA_DIR}/Lambda_Variations_and_Component_Ablations/{path_prefix}_Allele_Aware_Ablations_w_Alt_MHCPrime_Full_5_Seeds_Cleaned/",
]

df_names=["ms_val", "escape_val"]

loaded_dfs_all=load_and_merge_dir_dfs(results_dirs, df_names=df_names, key_cols=("seq", "allele", "label"), how="outer")

ms_val_for_escape_con=loaded_dfs_all["ms_val"]
escape_con_val=loaded_dfs_all["escape_val"]

# match the ms val to the assay val
ms_val_for_escape_con_matched, escape_con_val=filter_by_allele_len_pairs(ms_val_for_escape_con, escape_con_val, allele_col="allele", seq_col="seq", filter_both=False)

# create dfs dict
escape_con_dfs_aa_off={
    # "ms_val": ms_val_for_escape_con,
    "ms_val_matched": ms_val_for_escape_con_matched,
    "assay_val": escape_con_val,
}

for df_name, df in escape_con_dfs_aa_off.items():
    print(df_name)
    print_data_stats(df)
    print(end="\n")

[/home/pchati/PepCL_Manuscript/Data/Lambda_Variations_and_Component_Ablations/ESCAPE_seq_CON_Allele_Aware_Ablations_w_Alt_MHCPrime_Full_5_Seeds_Cleaned/] found 2 df(s) (filtered to 2 requested)


dir 0: ESCAPE_seq_CON_Allele_Aware_Ablations_w_Alt_MHCPrime_Full_5_Seeds_Cleaned:   0%|          | 0/2 [00:00<?, ?it/s]

dir 0: ESCAPE_seq_CON_Allele_Aware_Ablations_w_Alt_MHCPrime_Full_5_Seeds_Cleaned: 100%|██████████| 2/2 [00:00<00:00,  4.05it/s]


total unique dfs in store: 2
ms_val_matched
Total: 185,000
Pos: 1,850 | 1.00%
Neg: 183,150 | 99.00%
N alleles: 37

assay_val
Total: 1,315
Pos: 657 | 49.96%
Neg: 658 | 50.04%
N alleles: 41

CPU times: user 746 ms, sys: 87.3 ms, total: 833 ms
Wall time: 828 ms


In [ ]:
# get datasplits

def merge_keep_first_dups(df1, df2, key_cols=("seq","allele","label"), how="outer"):
    key_cols = list(key_cols)
    dup_cols = [c for c in df2.columns if c in df1.columns and c not in key_cols]
    df2_trimmed = df2.drop(columns=dup_cols)
    return df1.merge(df2_trimmed, on=key_cols, how=how)

datasplits=[1, 2, 3, 4]

path_prefix="ESCAPE_seq_CON"

results_dirs=[
    f"{DATA_DIR}/Lambda_Variations_and_Component_Ablations/{path_prefix}_Allele_Aware_Ablations_w_Alt_MHCPrime_Cleaned_CV_Datasplit_{ds}/" for ds in datasplits
]


df_names=["ms_val", "escape_val"]

escape_con_dfs_aa_off_ds={}

for ds, rd in zip(tqdm(datasplits), results_dirs):

    seed_num=ds+4 # offset by 4
    ms_val=pd.read_csv(f"{rd}/ms_val.csv")

    escape_con_val=pd.read_csv(f"{rd}/escape_val.csv")

    ms_val, escape_con_val=filter_by_allele_len_pairs(ms_val, escape_con_val, allele_col="allele", seq_col="seq", filter_both=False)

    ms_val.columns=ms_val.columns.str.replace("seed_0", f"seed_{seed_num}")
    escape_con_val.columns=escape_con_val.columns.str.replace("seed_0", f"seed_{seed_num}")
    escape_con_dfs_aa_off_ds[f"ms_val_matched_{ds}"]=ms_val
    escape_con_dfs_aa_off_ds[f"assay_val_{ds}"]=escape_con_val

100%|██████████| 4/4 [00:01<00:00,  2.01it/s]


### Compute performances

#### Get score columns

In [ ]:
"""
NOTE: create a list of dataset prefixes, and just create everything that way, easier to store.
"""

# get score cols (for pepcl_r_dr and our default settings for now)
dr_lambda_list=[0, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2]
dr_lambda_strings=lambdas_to_strings(dr_lambda_list)

default_steps=300
learning_rate_string="2eN4"
# pepcl_seed=0
pepcl_seeds=[0, 1, 2, 3, 4, 5, 6, 7, 8]

dataset_prefixes=["holec", "episcan_r9", "episcan_r9rt", "ba_con", "escape_con"]

score_cols_per_dataset={}
method_prefixes_per_dataset={}

for dataset_prefix in dataset_prefixes:
    score_columns=[]

    for pepcl_seed in pepcl_seeds:
        for dr_lambda_string in dr_lambda_strings:
            pepcl_r_dr_model_name=f"{dataset_prefix}_pepcl_r_dr_steps_{str(default_steps)}_lr_{learning_rate_string}_dr_lambda_{dr_lambda_string}_seed_{str(pepcl_seed)}"

            pepcl_dr_model_name=f"{dataset_prefix}_pepcl_dr_steps_{str(default_steps)}_lr_{learning_rate_string}_dr_lambda_{dr_lambda_string}_seed_{str(pepcl_seed)}"

            score_columns.append(pepcl_r_dr_model_name)
            score_columns.append(pepcl_dr_model_name)

    method_prefixes={
        "pepcl_r_dr": f"{dataset_prefix}_pepcl_r_dr_",
        # optionally:
        "pepcl_dr": f"{dataset_prefix}_pepcl_dr_",
    }

    score_cols_per_dataset[dataset_prefix]=score_columns
    method_prefixes_per_dataset[dataset_prefix]=method_prefixes

#### YD experiments

In [ ]:
def rename_mhcprime_by_datasplit(df_dict, ds_to_seed_offset=4):
    out={}

    for key,df in df_dict.items():
        df=df.copy()

        # assumes key ends in _1, _2, _3, _4
        ds=int(key.rsplit("_",1)[-1])
        seed=ds+ds_to_seed_offset

        if "mhcprime" in df.columns:
            df=df.rename(columns={"mhcprime":f"mhcprime_seed_{seed}"})

        out[key]=df

    return out

In [ ]:
# holec ap matrices
holec_ap_dfs={}

for df_name, df in tqdm(holec_dfs.items()):
    score_cols=["mhcprime"] + score_cols_per_dataset["holec"]
    ap_df=compute_ap_matrix_optimized(df, score_cols, label_col='label', allele_col='allele', disable_tqdm=True)
    holec_ap_dfs[df_name]=ap_df

holec_ap_dfs_ds={}

for df_name, df in tqdm(holec_dfs_ds.items()):
    score_cols=["mhcprime"] + score_cols_per_dataset["holec"]
    ap_df=compute_ap_matrix_optimized(df, score_cols, label_col='label', allele_col='allele', disable_tqdm=True)
    holec_ap_dfs_ds[df_name]=ap_df

# merge dataset splits ap dfs
holec_ap_dfs_ds=rename_mhcprime_by_datasplit(holec_ap_dfs_ds, ds_to_seed_offset=4)

holec_ap_dfs_ds_merged=merge_dfs_by_prefix(holec_ap_dfs_ds, ["ms_val_matched", "assay_val"])

# create full ap df
holec_ap_dfs=merge_two_dicts_by_keys(holec_ap_dfs, holec_ap_dfs_ds_merged, ["ms_val_matched", "assay_val"])

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 8/8 [00:02<00:00,  3.57it/s]


In [ ]:
# holec ap matrices
holec_ap_dfs_aa_off={}

for df_name, df in tqdm(holec_dfs_aa_off.items()):
    score_cols=["mhcprime"] + score_cols_per_dataset["holec"]
    ap_df=compute_ap_matrix_optimized(df, score_cols, label_col='label', allele_col='allele', disable_tqdm=True)
    holec_ap_dfs_aa_off[df_name]=ap_df

holec_ap_dfs_aa_off_ds={}

for df_name, df in tqdm(holec_dfs_aa_off_ds.items()):
    score_cols=["mhcprime"] + score_cols_per_dataset["holec"]
    ap_df=compute_ap_matrix_optimized(df, score_cols, label_col='label', allele_col='allele', disable_tqdm=True)
    holec_ap_dfs_aa_off_ds[df_name]=ap_df

# merge dataset splits ap dfs
holec_ap_dfs_aa_off_ds=rename_mhcprime_by_datasplit(holec_ap_dfs_aa_off_ds, ds_to_seed_offset=4)

holec_ap_dfs_aa_off_ds_merged=merge_dfs_by_prefix(holec_ap_dfs_aa_off_ds, ["ms_val_matched", "assay_val"])

# create full ap df
holec_ap_dfs_aa_off=merge_two_dicts_by_keys(holec_ap_dfs_aa_off, holec_ap_dfs_aa_off_ds_merged, ["ms_val_matched", "assay_val"])

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 8/8 [00:00<00:00, 11.18it/s]


#### EpiScan experiments

##### R9

In [ ]:
%%time
# episcan r9 aps
datasplits=[1, 2, 3, 4]

episcan_r9_ap_dfs={}

for df_name, df in tqdm(episcan_r9_dfs.items()):
    score_cols=["mhcprime"] + score_cols_per_dataset["episcan_r9"]
    ap_df = compute_ap_matrix_optimized(df, score_cols, label_col='label', allele_col='allele', disable_tqdm=True)
    episcan_r9_ap_dfs[df_name] = ap_df

episcan_r9_ap_dfs_ds={}

for df_name, df in tqdm(episcan_r9_dfs_ds.items()):
    score_cols=["mhcprime"] + score_cols_per_dataset["episcan_r9"]
    ap_df=compute_ap_matrix_optimized(df, score_cols, label_col='label', allele_col='allele', disable_tqdm=True)
    episcan_r9_ap_dfs_ds[df_name]=ap_df

n_iterations=100
allele_pos_counts = {"A0201": 100, "A0301": 100, "B0801": 100, "B5701": 100}

# per-allele AP matrix, pos-focused (negs from r9 only)
print("EpiScan R9")
episcan_r9_val_set_for_r9=episcan_r9_dfs["assay_val"].query("dataset_source == 'r9'")
episcan_rt_val_set_for_r9=episcan_r9_dfs["assay_val"].query("dataset_source == 'rt'")

score_cols=["mhcprime"] + score_cols_per_dataset["episcan_r9"]
score_cols=keep_present_cols(episcan_r9_dfs["assay_val"], score_cols)

r9rt_pos_r9_neg_per_allele_ap_for_r9 = mixed_df_per_allele_ap_pos_focused_fast(episcan_r9_val_set_for_r9, episcan_rt_val_set_for_r9.query("label == 1"), score_cols, allele_pos_counts, label_col1='label', label_col2='label', allele_col='allele', neg_ratio=99, n_iterations=n_iterations, seed=42, chunk_size=256, num_threads=16)

episcan_r9_ap_dfs["assay_val_mixture"]=r9rt_pos_r9_neg_per_allele_ap_for_r9

# repeat for datasplits
for ds in datasplits:
    # per-allele AP matrix, pos-focused (negs from r9 only)
    print("EpiScan R9")
    episcan_r9_val_set_for_r9=episcan_r9_dfs_ds[f"assay_val_{ds}"].query("dataset_source == 'r9'")
    episcan_rt_val_set_for_r9=episcan_r9_dfs_ds[f"assay_val_{ds}"].query("dataset_source == 'rt'")

    score_cols=["mhcprime"] + score_cols_per_dataset["episcan_r9"]
    score_cols=keep_present_cols(episcan_r9_dfs_ds[f"assay_val_{ds}"], score_cols)

    r9rt_pos_r9_neg_per_allele_ap_for_r9 = mixed_df_per_allele_ap_pos_focused_fast(episcan_r9_val_set_for_r9, episcan_rt_val_set_for_r9.query("label == 1"), score_cols, allele_pos_counts, label_col1='label', label_col2='label', allele_col='allele', neg_ratio=99, n_iterations=n_iterations, seed=42, chunk_size=256, num_threads=16)

    episcan_r9_ap_dfs_ds[f"assay_val_mixture_{ds}"]=r9rt_pos_r9_neg_per_allele_ap_for_r9

# merge dataset splits ap dfs
episcan_r9_ap_dfs_ds=rename_mhcprime_by_datasplit(episcan_r9_ap_dfs_ds, ds_to_seed_offset=4)

episcan_r9_ap_dfs_ds_merged=merge_dfs_by_prefix(episcan_r9_ap_dfs_ds, ["ms_val_matched", "assay_val_mixture"])

# create full ap df
episcan_r9_ap_dfs=merge_two_dicts_by_keys(episcan_r9_ap_dfs, episcan_r9_ap_dfs_ds_merged, ["ms_val_matched", "assay_val_mixture"])

100%|██████████| 8/8 [00:01<00:00,  6.61it/s]


EpiScan R9
[WARNING] Allele A0301 has only 83 positives (df1=33, df2=50), below requested 100
[WARNING] Allele B0801 has only 55 positives (df1=5, df2=50), below requested 100


Iterations: 100%|██████████| 100/100 [00:05<00:00, 18.53it/s]


EpiScan R9
[WARNING] Allele A0301 has only 83 positives (df1=33, df2=50), below requested 100
[WARNING] Allele B0801 has only 55 positives (df1=5, df2=50), below requested 100


Iterations: 100%|██████████| 100/100 [00:01<00:00, 83.40it/s]


...

EpiScan R9
[WARNING] Allele A0301 has only 83 positives (df1=33, df2=50), below requested 100
[WARNING] Allele B0801 has only 55 positives (df1=5, df2=50), below requested 100


Iterations: 100%|██████████| 100/100 [00:01<00:00, 86.83it/s]


EpiScan R9
[WARNING] Allele A0301 has only 83 positives (df1=33, df2=50), below requested 100
[WARNING] Allele B0801 has only 55 positives (df1=5, df2=50), below requested 100


Iterations: 100%|██████████| 100/100 [00:01<00:00, 83.65it/s]

CPU times: user 1min 10s, sys: 1.64 s, total: 1min 12s
Wall time: 17.2 s


In [ ]:
%%time
# episcan r9 aps
datasplits=[1, 2, 3, 4]

episcan_r9_ap_dfs_aa_off={}

for df_name, df in tqdm(episcan_r9_dfs_aa_off.items()):
    score_cols=["mhcprime"] + score_cols_per_dataset["episcan_r9"]
    ap_df = compute_ap_matrix_optimized(df, score_cols, label_col='label', allele_col='allele', disable_tqdm=True)
    episcan_r9_ap_dfs_aa_off[df_name] = ap_df

episcan_r9_ap_dfs_aa_off_ds={}

for df_name, df in tqdm(episcan_r9_dfs_aa_off_ds.items()):
    score_cols=["mhcprime"] + score_cols_per_dataset["episcan_r9"]
    ap_df=compute_ap_matrix_optimized(df, score_cols, label_col='label', allele_col='allele', disable_tqdm=True)
    episcan_r9_ap_dfs_aa_off_ds[df_name]=ap_df

n_iterations=100
allele_pos_counts = {"A0201": 100, "A0301": 100, "B0801": 100, "B5701": 100}

# per-allele AP matrix, pos-focused (negs from r9 only)
print("EpiScan R9")
episcan_r9_val_set_for_r9=episcan_r9_dfs_aa_off["assay_val"].query("dataset_source == 'r9'")
episcan_rt_val_set_for_r9=episcan_r9_dfs_aa_off["assay_val"].query("dataset_source == 'rt'")

score_cols=["mhcprime"] + score_cols_per_dataset["episcan_r9"]
score_cols=keep_present_cols(episcan_r9_dfs_aa_off["assay_val"], score_cols)

r9rt_pos_r9_neg_per_allele_ap_for_r9 = mixed_df_per_allele_ap_pos_focused_fast(episcan_r9_val_set_for_r9, episcan_rt_val_set_for_r9.query("label == 1"), score_cols, allele_pos_counts, label_col1='label', label_col2='label', allele_col='allele', neg_ratio=99, n_iterations=n_iterations, seed=42, chunk_size=256, num_threads=16)

episcan_r9_ap_dfs_aa_off["assay_val_mixture"]=r9rt_pos_r9_neg_per_allele_ap_for_r9

# repeat for datasplits
for ds in datasplits:
    # per-allele AP matrix, pos-focused (negs from r9 only)
    print("EpiScan R9")
    episcan_r9_val_set_for_r9=episcan_r9_dfs_aa_off_ds[f"assay_val_{ds}"].query("dataset_source == 'r9'")
    episcan_rt_val_set_for_r9=episcan_r9_dfs_aa_off_ds[f"assay_val_{ds}"].query("dataset_source == 'rt'")

    score_cols=["mhcprime"] + score_cols_per_dataset["episcan_r9"]
    score_cols=keep_present_cols(episcan_r9_dfs_aa_off_ds[f"assay_val_{ds}"], score_cols)

    r9rt_pos_r9_neg_per_allele_ap_for_r9 = mixed_df_per_allele_ap_pos_focused_fast(episcan_r9_val_set_for_r9, episcan_rt_val_set_for_r9.query("label == 1"), score_cols, allele_pos_counts, label_col1='label', label_col2='label', allele_col='allele', neg_ratio=99, n_iterations=n_iterations, seed=42, chunk_size=256, num_threads=16)

    episcan_r9_ap_dfs_aa_off_ds[f"assay_val_mixture_{ds}"]=r9rt_pos_r9_neg_per_allele_ap_for_r9

# merge dataset splits ap dfs
episcan_r9_ap_dfs_aa_off_ds=rename_mhcprime_by_datasplit(episcan_r9_ap_dfs_aa_off_ds, ds_to_seed_offset=4)

episcan_r9_ap_dfs_aa_off_ds_merged=merge_dfs_by_prefix(episcan_r9_ap_dfs_aa_off_ds, ["ms_val_matched", "assay_val_mixture"])

# create full ap df
episcan_r9_ap_dfs_aa_off=merge_two_dicts_by_keys(episcan_r9_ap_dfs_aa_off, episcan_r9_ap_dfs_aa_off_ds_merged, ["ms_val_matched", "assay_val_mixture"])

  0%|          | 0/4 [00:00<?, ?it/s]

100%|██████████| 8/8 [00:00<00:00, 17.40it/s]


EpiScan R9
[WARNING] Allele A0301 has only 83 positives (df1=33, df2=50), below requested 100
[WARNING] Allele B0801 has only 55 positives (df1=5, df2=50), below requested 100


Iterations: 100%|██████████| 100/100 [00:01<00:00, 56.41it/s]


EpiScan R9
[WARNING] Allele A0301 has only 83 positives (df1=33, df2=50), below requested 100
[WARNING] Allele B0801 has only 55 positives (df1=5, df2=50), below requested 100


...

EpiScan R9
[WARNING] Allele A0301 has only 83 positives (df1=33, df2=50), below requested 100
[WARNING] Allele B0801 has only 55 positives (df1=5, df2=50), below requested 100


Iterations: 100%|██████████| 100/100 [00:00<00:00, 102.75it/s]


EpiScan R9
[WARNING] Allele A0301 has only 83 positives (df1=33, df2=50), below requested 100
[WARNING] Allele B0801 has only 55 positives (df1=5, df2=50), below requested 100


Iterations: 100%|██████████| 100/100 [00:00<00:00, 103.45it/s]

CPU times: user 21.2 s, sys: 2.2 s, total: 23.4 s
Wall time: 7.1 s


##### R9/RT

In [ ]:
%%time
# episcan r9rt aps
datasplits=[1, 2, 3, 4]

episcan_r9rt_ap_dfs={}

for df_name, df in tqdm(episcan_r9rt_dfs.items()):
    score_cols=["mhcprime"] + score_cols_per_dataset["episcan_r9rt"]
    ap_df = compute_ap_matrix_optimized(df, score_cols, label_col='label', allele_col='allele', disable_tqdm=True)
    episcan_r9rt_ap_dfs[df_name] = ap_df

episcan_r9rt_ap_dfs_ds={}

for df_name, df in tqdm(episcan_r9rt_dfs_ds.items()):
    score_cols=["mhcprime"] + score_cols_per_dataset["episcan_r9rt"]
    ap_df=compute_ap_matrix_optimized(df, score_cols, label_col='label', allele_col='allele', disable_tqdm=True)
    episcan_r9rt_ap_dfs_ds[df_name]=ap_df

n_iterations=100
allele_pos_counts = {"A0201": 100, "A0301": 100, "B0801": 100, "B5701": 100}

# per-allele AP matrix, both (negs split across r9 and rt)
print("EpiScan R9/RT")
episcan_r9_val_set_for_r9rt=episcan_r9rt_dfs["assay_val"].query("dataset_source == 'r9'")
episcan_rt_val_set_for_r9rt=episcan_r9rt_dfs["assay_val"].query("dataset_source == 'rt'")

score_cols=["mhcprime"] + score_cols_per_dataset["episcan_r9rt"]
score_cols=keep_present_cols(episcan_r9rt_dfs["assay_val"], score_cols)

r9rt_pos_r9rt_neg_per_allele_ap_for_r9rt = mixed_df_per_allele_ap_balanced_fast(episcan_r9_val_set_for_r9rt, episcan_rt_val_set_for_r9rt, score_cols, label_col1='label', label_col2='label', allele_col='allele', neg_ratio=99, n_iterations=n_iterations, seed=42, chunk_size=256, num_threads=16)


episcan_r9rt_ap_dfs["assay_val_mixture"]=r9rt_pos_r9rt_neg_per_allele_ap_for_r9rt

# repeat for datasplits
for ds in datasplits:
    # per-allele AP matrix, both (negs split across r9 and rt)
    print("EpiScan R9/RT")
    episcan_r9_val_set_for_r9rt=episcan_r9rt_dfs_ds[f"assay_val_{ds}"].query("dataset_source == 'r9'")
    episcan_rt_val_set_for_r9rt=episcan_r9rt_dfs_ds[f"assay_val_{ds}"].query("dataset_source == 'rt'")

    # score_cols = ["mhcprime"] + list(episcan_r9rt_model_groups.keys())
    score_cols=["mhcprime"] + score_cols_per_dataset["episcan_r9rt"]
    score_cols=keep_present_cols(episcan_r9rt_dfs_ds[f"assay_val_{ds}"], score_cols)

    r9rt_pos_r9rt_neg_per_allele_ap_for_r9rt = mixed_df_per_allele_ap_balanced_fast(episcan_r9_val_set_for_r9rt, episcan_rt_val_set_for_r9rt, score_cols, label_col1='label', label_col2='label', allele_col='allele', neg_ratio=99, n_iterations=n_iterations, seed=42, chunk_size=256, num_threads=16)

    episcan_r9rt_ap_dfs_ds[f"assay_val_mixture_{ds}"]=r9rt_pos_r9rt_neg_per_allele_ap_for_r9rt

# merge dataset splits ap dfs
episcan_r9rt_ap_dfs_ds=rename_mhcprime_by_datasplit(episcan_r9rt_ap_dfs_ds, ds_to_seed_offset=4)

episcan_r9rt_ap_dfs_ds_merged=merge_dfs_by_prefix(episcan_r9rt_ap_dfs_ds, ["ms_val_matched", "assay_val_mixture"])

# create full ap df
episcan_r9rt_ap_dfs=merge_two_dicts_by_keys(episcan_r9rt_ap_dfs, episcan_r9rt_ap_dfs_ds_merged, ["ms_val_matched", "assay_val_mixture"])

100%|██████████| 8/8 [00:01<00:00,  5.47it/s]


EpiScan R9/RT


Iterations: 100%|██████████| 100/100 [00:07<00:00, 14.22it/s]


EpiScan R9/RT


Iterations: 100%|██████████| 100/100 [00:02<00:00, 48.56it/s]


...

EpiScan R9/RT


Iterations: 100%|██████████| 100/100 [00:02<00:00, 48.43it/s]


EpiScan R9/RT


Iterations: 100%|██████████| 100/100 [00:01<00:00, 50.10it/s]

CPU times: user 2min 18s, sys: 3.07 s, total: 2min 21s
Wall time: 19.9 s


In [ ]:
%%time
# episcan r9rt aps
datasplits=[1, 2, 3, 4]

episcan_r9rt_ap_dfs_aa_off={}

for df_name, df in tqdm(episcan_r9rt_dfs_aa_off.items()):
    score_cols=["mhcprime"] + score_cols_per_dataset["episcan_r9rt"]
    ap_df = compute_ap_matrix_optimized(df, score_cols, label_col='label', allele_col='allele', disable_tqdm=True)
    episcan_r9rt_ap_dfs_aa_off[df_name] = ap_df

episcan_r9rt_ap_dfs_aa_off_ds={}

for df_name, df in tqdm(episcan_r9rt_dfs_aa_off_ds.items()):
    score_cols=["mhcprime"] + score_cols_per_dataset["episcan_r9rt"]
    ap_df=compute_ap_matrix_optimized(df, score_cols, label_col='label', allele_col='allele', disable_tqdm=True)
    episcan_r9rt_ap_dfs_aa_off_ds[df_name]=ap_df

n_iterations=100
allele_pos_counts = {"A0201": 100, "A0301": 100, "B0801": 100, "B5701": 100}

# per-allele AP matrix, both (negs split across r9 and rt)
print("EpiScan R9/RT")
episcan_r9_val_set_for_r9rt=episcan_r9rt_dfs_aa_off["assay_val"].query("dataset_source == 'r9'")
episcan_rt_val_set_for_r9rt=episcan_r9rt_dfs_aa_off["assay_val"].query("dataset_source == 'rt'")

score_cols=["mhcprime"] + score_cols_per_dataset["episcan_r9rt"]
score_cols=keep_present_cols(episcan_r9rt_dfs_aa_off["assay_val"], score_cols)

r9rt_pos_r9rt_neg_per_allele_ap_for_r9rt = mixed_df_per_allele_ap_balanced_fast(episcan_r9_val_set_for_r9rt, episcan_rt_val_set_for_r9rt, score_cols, label_col1='label', label_col2='label', allele_col='allele', neg_ratio=99, n_iterations=n_iterations, seed=42, chunk_size=256, num_threads=16)


episcan_r9rt_ap_dfs_aa_off["assay_val_mixture"]=r9rt_pos_r9rt_neg_per_allele_ap_for_r9rt

# repeat for datasplits
for ds in datasplits:
    # per-allele AP matrix, both (negs split across r9 and rt)
    print("EpiScan R9/RT")
    episcan_r9_val_set_for_r9rt=episcan_r9rt_dfs_aa_off_ds[f"assay_val_{ds}"].query("dataset_source == 'r9'")
    episcan_rt_val_set_for_r9rt=episcan_r9rt_dfs_aa_off_ds[f"assay_val_{ds}"].query("dataset_source == 'rt'")

    # score_cols = ["mhcprime"] + list(episcan_r9rt_model_groups.keys())
    score_cols=["mhcprime"] + score_cols_per_dataset["episcan_r9rt"]
    score_cols=keep_present_cols(episcan_r9rt_dfs_aa_off_ds[f"assay_val_{ds}"], score_cols)

    r9rt_pos_r9rt_neg_per_allele_ap_for_r9rt = mixed_df_per_allele_ap_balanced_fast(episcan_r9_val_set_for_r9rt, episcan_rt_val_set_for_r9rt, score_cols, label_col1='label', label_col2='label', allele_col='allele', neg_ratio=99, n_iterations=n_iterations, seed=42, chunk_size=256, num_threads=16)

    episcan_r9rt_ap_dfs_aa_off_ds[f"assay_val_mixture_{ds}"]=r9rt_pos_r9rt_neg_per_allele_ap_for_r9rt

# merge dataset splits ap dfs
episcan_r9rt_ap_dfs_aa_off_ds=rename_mhcprime_by_datasplit(episcan_r9rt_ap_dfs_aa_off_ds, ds_to_seed_offset=4)

episcan_r9rt_ap_dfs_aa_off_ds_merged=merge_dfs_by_prefix(episcan_r9rt_ap_dfs_aa_off_ds, ["ms_val_matched", "assay_val_mixture"])

# create full ap df
episcan_r9rt_ap_dfs_aa_off=merge_two_dicts_by_keys(episcan_r9rt_ap_dfs_aa_off, episcan_r9rt_ap_dfs_aa_off_ds_merged, ["ms_val_matched", "assay_val_mixture"])

100%|██████████| 8/8 [00:00<00:00, 16.95it/s]


EpiScan R9/RT


Iterations: 100%|██████████| 100/100 [00:03<00:00, 32.77it/s]


EpiScan R9/RT


Iterations: 100%|██████████| 100/100 [00:01<00:00, 58.18it/s]


...

EpiScan R9/RT


Iterations: 100%|██████████| 100/100 [00:01<00:00, 59.73it/s]


EpiScan R9/RT


Iterations: 100%|██████████| 100/100 [00:01<00:00, 57.71it/s]

CPU times: user 42 s, sys: 2.28 s, total: 44.3 s
Wall time: 11.4 s


#### BA CON

In [ ]:
%%time
# ba aps
ba_con_ap_dfs={}

for df_name, df in tqdm(ba_con_dfs.items()):
    score_cols=["mhcprime"] + score_cols_per_dataset["ba_con"]
    if df_name == "ms_val" or df_name == "ms_val_matched":
        print(f"{df_name}: AP branch")
        ap_df = compute_ap_matrix_optimized(df, score_cols, label_col='label', allele_col='allele', disable_tqdm=True)
        ba_con_ap_dfs[df_name] = ap_df
    else:
        print(f"{df_name}: SPEARMAN branch")
        spearman_df = compute_spearman_matrix_optimized(df, score_cols, reference_col="ba_value", allele_col='allele', disable_tqdm=True)
        ba_con_ap_dfs[df_name] = spearman_df

        df["all"]=1
        spearman_df_all = compute_spearman_matrix_optimized(df, score_cols, reference_col="ba_value", allele_col='all', disable_tqdm=True)
        ba_con_ap_dfs["assay_val_all"] = spearman_df_all

        spearman_df_pos = compute_spearman_matrix_optimized(df.query("label == 1"), score_cols, reference_col="ba_value", allele_col='allele', disable_tqdm=True)
        ba_con_ap_dfs["assay_val_pos"] = spearman_df_pos

        spearman_df_all_pos = compute_spearman_matrix_optimized(df.query("label == 1"), score_cols, reference_col="ba_value", allele_col='all', disable_tqdm=True)
        ba_con_ap_dfs["assay_val_all_pos"] = spearman_df_all_pos

ba_con_ap_dfs_ds={}

for df_name, df in tqdm(ba_con_dfs_ds.items()):
    score_cols=["mhcprime"] + score_cols_per_dataset["ba_con"]
    if "ms_val_matched" in df_name:
        print(f"{df_name}: AP branch")
        ap_df = compute_ap_matrix_optimized(df, score_cols, label_col='label', allele_col='allele', disable_tqdm=True)
        ba_con_ap_dfs_ds[df_name] = ap_df
    else:
        print(f"{df_name}: SPEARMAN branch")
        spearman_df = compute_spearman_matrix_optimized(df, score_cols, reference_col="ba_value", allele_col='allele', disable_tqdm=True)
        ba_con_ap_dfs_ds[df_name] = spearman_df

# merge dataset splits ap dfs
ba_con_ap_dfs_ds=rename_mhcprime_by_datasplit(ba_con_ap_dfs_ds, ds_to_seed_offset=4)

ba_con_ap_dfs_ds_merged=merge_dfs_by_prefix(ba_con_ap_dfs_ds, ["ms_val_matched", "assay_val"])

# create full ap df
ba_con_ap_dfs=merge_two_dicts_by_keys(ba_con_ap_dfs, ba_con_ap_dfs_ds_merged, ["ms_val_matched", "assay_val"])

  0%|          | 0/2 [00:00<?, ?it/s]

ms_val_matched: AP branch


 50%|█████     | 1/2 [00:09<00:09,  9.28s/it]

assay_val: SPEARMAN branch


  0%|          | 0/8 [00:00<?, ?it/s]

...

ms_val_matched_4: AP branch


 88%|████████▊ | 7/8 [00:06<00:00,  1.09it/s]

assay_val_4: SPEARMAN branch


100%|██████████| 8/8 [00:07<00:00,  1.14it/s]

CPU times: user 22 s, sys: 145 ms, total: 22.1 s
Wall time: 22.1 s


In [ ]:
%%time
# ba aps
ba_con_ap_dfs_aa_off={}

for df_name, df in tqdm(ba_con_dfs_aa_off.items()):
    score_cols=["mhcprime"] + score_cols_per_dataset["ba_con"]
    if df_name == "ms_val" or df_name == "ms_val_matched":
        print(f"{df_name}: AP branch")
        ap_df = compute_ap_matrix_optimized(df, score_cols, label_col='label', allele_col='allele', disable_tqdm=True)
        ba_con_ap_dfs_aa_off[df_name] = ap_df
    else:
        print(f"{df_name}: SPEARMAN branch")
        spearman_df = compute_spearman_matrix_optimized(df, score_cols, reference_col="ba_value", allele_col='allele', disable_tqdm=True)
        ba_con_ap_dfs_aa_off[df_name] = spearman_df

        df["all"]=1
        spearman_df_all = compute_spearman_matrix_optimized(df, score_cols, reference_col="ba_value", allele_col='all', disable_tqdm=True)
        ba_con_ap_dfs_aa_off["assay_val_all"] = spearman_df_all

        spearman_df_pos = compute_spearman_matrix_optimized(df.query("label == 1"), score_cols, reference_col="ba_value", allele_col='allele', disable_tqdm=True)
        ba_con_ap_dfs_aa_off["assay_val_pos"] = spearman_df_pos

        spearman_df_all_pos = compute_spearman_matrix_optimized(df.query("label == 1"), score_cols, reference_col="ba_value", allele_col='all', disable_tqdm=True)
        ba_con_ap_dfs_aa_off["assay_val_all_pos"] = spearman_df_all_pos

ba_con_ap_dfs_aa_off_ds={}

for df_name, df in tqdm(ba_con_dfs_aa_off_ds.items()):
    score_cols=["mhcprime"] + score_cols_per_dataset["ba_con"]
    if "ms_val_matched" in df_name:
        print(f"{df_name}: AP branch")
        ap_df = compute_ap_matrix_optimized(df, score_cols, label_col='label', allele_col='allele', disable_tqdm=True)
        ba_con_ap_dfs_aa_off_ds[df_name] = ap_df
    else:
        print(f"{df_name}: SPEARMAN branch")
        spearman_df = compute_spearman_matrix_optimized(df, score_cols, reference_col="ba_value", allele_col='allele', disable_tqdm=True)
        ba_con_ap_dfs_aa_off_ds[df_name] = spearman_df

# merge dataset splits ap dfs
ba_con_ap_dfs_aa_off_ds=rename_mhcprime_by_datasplit(ba_con_ap_dfs_aa_off_ds, ds_to_seed_offset=4)

ba_con_ap_dfs_aa_off_ds_merged=merge_dfs_by_prefix(ba_con_ap_dfs_aa_off_ds, ["ms_val_matched", "assay_val"])

# create full ap df
ba_con_ap_dfs_aa_off=merge_two_dicts_by_keys(ba_con_ap_dfs_aa_off, ba_con_ap_dfs_aa_off_ds_merged, ["ms_val_matched", "assay_val"])

  0%|          | 0/2 [00:00<?, ?it/s]

ms_val_matched: AP branch


 50%|█████     | 1/2 [00:01<00:01,  1.37s/it]

assay_val: SPEARMAN branch


  0%|          | 0/8 [00:00<?, ?it/s]

...

assay_val_2: SPEARMAN branch
ms_val_matched_3: AP branch


 75%|███████▌  | 6/8 [00:01<00:00,  4.00it/s]

assay_val_3: SPEARMAN branch
ms_val_matched_4: AP branch


100%|██████████| 8/8 [00:02<00:00,  3.78it/s]

assay_val_4: SPEARMAN branch
CPU times: user 5.01 s, sys: 39.9 ms, total: 5.05 s
Wall time: 5.03 s


#### ESCAPE CON

In [ ]:
%%time
# escape aps
escape_con_ap_dfs={}

for df_name, df in tqdm(escape_con_dfs.items()):
    score_cols=["mhcprime"] + score_cols_per_dataset["escape_con"]
    if df_name == "ms_val" or df_name == "ms_val_matched":
        print(f"{df_name}: AP branch")
        ap_df = compute_ap_matrix_optimized(df, score_cols, label_col='label', allele_col='allele', disable_tqdm=True)
        escape_con_ap_dfs[df_name] = ap_df
    else:
        print(f"{df_name}: SPEARMAN branch")
        spearman_df = compute_spearman_matrix_optimized(df, score_cols, reference_col="escore", allele_col='allele', disable_tqdm=True)
        escape_con_ap_dfs[df_name] = spearman_df

        df["all"]=1
        spearman_df_all = compute_spearman_matrix_optimized(df, score_cols, reference_col="escore", allele_col='all', disable_tqdm=True)
        escape_con_ap_dfs["assay_val_all"] = spearman_df_all

        spearman_df_pos = compute_spearman_matrix_optimized(df.query("label == 1"), score_cols, reference_col="escore", allele_col='allele', disable_tqdm=True)
        escape_con_ap_dfs["assay_val_pos"] = spearman_df_pos

        spearman_df_all_pos = compute_spearman_matrix_optimized(df.query("label == 1"), score_cols, reference_col="escore", allele_col='all', disable_tqdm=True)
        escape_con_ap_dfs["assay_val_all_pos"] = spearman_df_all_pos


escape_con_ap_dfs_ds={}

for df_name, df in tqdm(escape_con_dfs_ds.items()):
    score_cols=["mhcprime"] + score_cols_per_dataset["escape_con"]
    if "ms_val_matched" in df_name:
        print(f"{df_name}: AP branch")
        ap_df = compute_ap_matrix_optimized(df, score_cols, label_col='label', allele_col='allele', disable_tqdm=True)
        escape_con_ap_dfs_ds[df_name] = ap_df
    else:
        print(f"{df_name}: SPEARMAN branch")
        spearman_df = compute_spearman_matrix_optimized(df, score_cols, reference_col="escore", allele_col='allele', disable_tqdm=True)
        escape_con_ap_dfs_ds[df_name] = spearman_df

# merge dataset splits ap dfs
escape_con_ap_dfs_ds=rename_mhcprime_by_datasplit(escape_con_ap_dfs_ds, ds_to_seed_offset=4)

escape_con_ap_dfs_ds_merged=merge_dfs_by_prefix(escape_con_ap_dfs_ds, ["ms_val_matched", "assay_val"])

# create full ap df
escape_con_ap_dfs=merge_two_dicts_by_keys(escape_con_ap_dfs, escape_con_ap_dfs_ds_merged, ["ms_val_matched", "assay_val"])

  0%|          | 0/2 [00:00<?, ?it/s]

ms_val_matched: AP branch


 50%|█████     | 1/2 [00:02<00:02,  2.24s/it]

assay_val: SPEARMAN branch


  0%|          | 0/8 [00:00<?, ?it/s]

...

ms_val_matched_4: AP branch


 88%|████████▊ | 7/8 [00:03<00:00,  2.23it/s]

assay_val_4: SPEARMAN branch


100%|██████████| 8/8 [00:03<00:00,  2.27it/s]

CPU times: user 9.44 s, sys: 35.6 ms, total: 9.48 s
Wall time: 9.46 s


In [ ]:
%%time
# escape aps
escape_con_ap_dfs_aa_off={}

for df_name, df in tqdm(escape_con_dfs_aa_off.items()):
    score_cols=["mhcprime"] + score_cols_per_dataset["escape_con"]
    if df_name == "ms_val" or df_name == "ms_val_matched":
        print(f"{df_name}: AP branch")
        ap_df = compute_ap_matrix_optimized(df, score_cols, label_col='label', allele_col='allele', disable_tqdm=True)
        escape_con_ap_dfs_aa_off[df_name] = ap_df
    else:
        print(f"{df_name}: SPEARMAN branch")
        spearman_df = compute_spearman_matrix_optimized(df, score_cols, reference_col="escore", allele_col='allele', disable_tqdm=True)
        escape_con_ap_dfs_aa_off[df_name] = spearman_df

        df["all"]=1
        spearman_df_all = compute_spearman_matrix_optimized(df, score_cols, reference_col="escore", allele_col='all', disable_tqdm=True)
        escape_con_ap_dfs_aa_off["assay_val_all"] = spearman_df_all

        spearman_df_pos = compute_spearman_matrix_optimized(df.query("label == 1"), score_cols, reference_col="escore", allele_col='allele', disable_tqdm=True)
        escape_con_ap_dfs_aa_off["assay_val_pos"] = spearman_df_pos

        spearman_df_all_pos = compute_spearman_matrix_optimized(df.query("label == 1"), score_cols, reference_col="escore", allele_col='all', disable_tqdm=True)
        escape_con_ap_dfs_aa_off["assay_val_all_pos"] = spearman_df_all_pos


escape_con_ap_dfs_aa_off_ds={}

for df_name, df in tqdm(escape_con_dfs_aa_off_ds.items()):
    score_cols=["mhcprime"] + score_cols_per_dataset["escape_con"]
    if "ms_val_matched" in df_name:
        print(f"{df_name}: AP branch")
        ap_df = compute_ap_matrix_optimized(df, score_cols, label_col='label', allele_col='allele', disable_tqdm=True)
        escape_con_ap_dfs_aa_off_ds[df_name] = ap_df
    else:
        print(f"{df_name}: SPEARMAN branch")
        spearman_df = compute_spearman_matrix_optimized(df, score_cols, reference_col="escore", allele_col='allele', disable_tqdm=True)
        escape_con_ap_dfs_aa_off_ds[df_name] = spearman_df

# merge dataset splits ap dfs
escape_con_ap_dfs_aa_off_ds=rename_mhcprime_by_datasplit(escape_con_ap_dfs_aa_off_ds, ds_to_seed_offset=4)

escape_con_ap_dfs_aa_off_ds_merged=merge_dfs_by_prefix(escape_con_ap_dfs_aa_off_ds, ["ms_val_matched", "assay_val"])

# create full ap df
escape_con_ap_dfs_aa_off=merge_two_dicts_by_keys(escape_con_ap_dfs_aa_off, escape_con_ap_dfs_aa_off_ds_merged, ["ms_val_matched", "assay_val"])

  0%|          | 0/2 [00:00<?, ?it/s]

ms_val_matched: AP branch


 50%|█████     | 1/2 [00:00<00:00,  1.19it/s]

assay_val: SPEARMAN branch


  0%|          | 0/8 [00:00<?, ?it/s]

...

assay_val_2: SPEARMAN branch
ms_val_matched_3: AP branch


 75%|███████▌  | 6/8 [00:01<00:00,  5.30it/s]

assay_val_3: SPEARMAN branch
ms_val_matched_4: AP branch


100%|██████████| 8/8 [00:01<00:00,  5.19it/s]

assay_val_4: SPEARMAN branch
CPU times: user 3.88 s, sys: 19.7 ms, total: 3.9 s
Wall time: 3.88 s


### Save results for plotting

In [ ]:
# save data
lambda_colors={
    "0":"#081470",
    "1eN8":"#3057A6",
    "1eN7":"#6C8FD3",
    "1eN6":"#9BB0ED",
    "1eN5":"#37837C",
    "1eN4":"#7DCBBB",
    "1eN3":"#FFB55C",
    "1eN2":"#FFDEA4",
}

lambda_plot_data={
    "general":{
        "score_cols_per_dataset":score_cols_per_dataset,
        "method_prefixes_per_dataset":method_prefixes_per_dataset,
        "lambda_colors":lambda_colors,

        "method":"pepcl_r_dr",
        "metric1":"ms_decrease_vs_base",
        "metric2":"assay_gain_vs_base",
        "errors":("ci95","ci95","ci95"),

        "highlight_lambda":None,
        "crosshair_lambda":1e-5,
        "crosshair_modes":("both","vertical","vertical"),

        "xlabels":(
            "MS ΔmAP vs MHCPrime",
            "DR λ",
            "DR λ",
        ),

        "metric_line_colors":("black","black"),
        "metric_linewidth":1.0,
        "metric_marker_size":7,
        "metric_color_by_lambda":False,

        "pareto_marker_size":85,
        "highlight_size":220,

        "point_alpha":1.0,
        "metric_point_alpha":1.0,
        "pareto_error_alpha":1.0,
        "metric_error_alpha":1.0,

        "tick_decimals":2,
        "trim_tick_zeros":False,
        "n_ticks":5,
        "spine_width":1.0,
        "tick_width":1.0,
        "grid":False,

        "fig_size":(16,6),
        "dpi":300,
        "square_panels":True,

        "wspace":0.42,
        "middle_panel_shift":0.10,

        "colorbar_position":"pareto",
        "colorbar_size":0.035,
        "colorbar_pad":0.05,
    },

    "holec":{
        "ap_dfs":holec_ap_dfs,
        "assay_key":"assay_val",
        "ms_key":"ms_val_matched",
        "per_allele":False,
        "titles":(
            "Adaptation-retention",
            "MS retention",
            "YD adaptation",
        ),
        "ylabels":(
            "YD ΔmAP vs MHCPrime",
            "MS decrease vs MHCPrime",
            "YD gain vs MHCPrime",
        ),
    },

    "episcan_r9":{
        "ap_dfs":episcan_r9_ap_dfs,
        "assay_key":"assay_val_mixture",
        "ms_key":"ms_val_matched",
        "per_allele":False,
        "titles":(
            "Adaptation-retention",
            "MS retention",
            "EpiScan R9 adaptation",
        ),
        "ylabels":(
            "EpiScan R9 ΔmAP vs MHCPrime",
            "MS decrease vs MHCPrime",
            "EpiScan R9 gain vs MHCPrime",
        ),
    },

    "episcan_r9rt":{
        "ap_dfs":episcan_r9rt_ap_dfs,
        "assay_key":"assay_val_mixture",
        "ms_key":"ms_val_matched",
        "per_allele":False,
        "titles":(
            "Adaptation-retention",
            "MS retention",
            "EpiScan R9/RT adaptation",
        ),
        "ylabels":(
            "EpiScan R9/RT ΔmAP vs MHCPrime",
            "MS decrease vs MHCPrime",
            "EpiScan R9/RT gain vs MHCPrime",
        ),
    },

    "ba_con":{
        "ap_dfs":ba_con_ap_dfs,
        "assay_key":"assay_val",
        "ms_key":"ms_val_matched",
        "per_allele":False,
        "titles":(
            "Adaptation-retention",
            "MS retention",
            "BA adaptation",
        ),
        "ylabels":(
            "BA Δmean Spearman vs MHCPrime",
            "MS decrease vs MHCPrime",
            "BA gain vs MHCPrime",
        ),
    },

    "escape_con":{
        "ap_dfs":escape_con_ap_dfs,
        "assay_key":"assay_val",
        "ms_key":"ms_val_matched",
        "per_allele":False,
        "titles":(
            "Adaptation-retention",
            "MS retention",
            "ESCAPE-seq adaptation",
        ),
        "ylabels":(
            "ESCAPE-seq ΔSpearman vs MHCPrime",
            "MS decrease vs MHCPrime",
            "ESCAPE-seq gain vs MHCPrime",
        ),
    },
}

# save_dict_pickle(lambda_plot_data, f"{DATA_DIR}/Revisions_Data/lambda_sensitivity_data_pareto_curves_results.pkl")

In [ ]:
# save for revisions
component_ablation_ap_dfs={
    "holec":{
        "aa_on":holec_ap_dfs,
        "aa_off":holec_ap_dfs_aa_off,
    },

    "episcan_r9":{
        "aa_on":episcan_r9_ap_dfs,
        "aa_off":episcan_r9_ap_dfs_aa_off,
    },

    "episcan_r9rt":{
        "aa_on":episcan_r9rt_ap_dfs,
        "aa_off":episcan_r9rt_ap_dfs_aa_off,
    },

    "ba_con":{
        "aa_on":ba_con_ap_dfs,
        "aa_off":ba_con_ap_dfs_aa_off,
    },

    "escape_con":{
        "aa_on":escape_con_ap_dfs,
        "aa_off":escape_con_ap_dfs_aa_off,
    },
    "general": {
        "score_cols_per_dataset": score_cols_per_dataset,
        "method_prefixes_per_dataset": method_prefixes_per_dataset,
    }
}
# save_dict_pickle(component_ablation_ap_dfs, f"{DATA_DIR}/Revisions_Data/component_ablations_results.pkl")

### Joint updating results

#### Load primary results

In [ ]:
%%time
# load data

# full ms
joint_data=load_joint_retraining_dataframes(
    dataset_names=["yd", "episcan_r9", "episcan_r9rt", "ba_con", "escape_con"],
    joint_retraining_path=DATA_DIR/"Joint_Retraining",
)

# matched ms
joint_data_ms_matched=load_joint_retraining_dataframes(
    dataset_names=["yd", "episcan_r9", "episcan_r9rt", "ba_con", "escape_con"],
    joint_retraining_path=DATA_DIR/"Joint_Retraining_MS_Matched",
)


yd:
  Joint_Retraining/yd/holec_test: (160000, 15)
  Joint_Retraining/yd/holec_val: (72300, 173)
  Joint_Retraining/yd/ms_test: (3214300, 15)
  Joint_Retraining/yd/ms_val: (2075300, 173)

episcan_r9:
  Joint_Retraining/episcan_r9/cov_test: (326579, 15)
  Joint_Retraining/episcan_r9/episcan_r9_rt_test: (80000, 16)
  Joint_Retraining/episcan_r9/episcan_r9_rt_val: (33800, 174)
  Joint_Retraining/episcan_r9/ms_test: (3214300, 15)
  Joint_Retraining/episcan_r9/ms_val: (2075300, 173)

episcan_r9rt:
  Joint_Retraining/episcan_r9rt/cov_test: (326579, 15)
  Joint_Retraining/episcan_r9rt/episcan_r9_rt_test: (80000, 16)
  Joint_Retraining/episcan_r9rt/episcan_r9_rt_val: (33800, 174)
  Joint_Retraining/episcan_r9rt/ms_test: (3214300, 15)
  Joint_Retraining/episcan_r9rt/ms_val: (2075300, 173)

ba_con:
  Joint_Retraining/ba_con/ba_test: (4221, 16)
  Joint_Retraining/ba_con/ba_val: (3052, 174)
  Joint_Retraining/ba_con/ms_test: (3214300, 15)
  Joint_Retraining/ba_con/ms_val: (2075300, 173)

escape_c

In [ ]:
%%time
# optimal points selected dfs for full and matched ms
joint_data_es=load_joint_retraining_dataframes(
    dataset_names=["yd", "episcan_r9", "episcan_r9rt", "ba_con", "escape_con"],
    joint_retraining_path=[DATA_DIR/"Joint_Retraining_Early_Stopping", DATA_DIR/"Joint_Retraining_Early_Stopping_Updated"]
)

joint_data_ms_matched_es=load_joint_retraining_dataframes(
    dataset_names=["yd", "episcan_r9", "episcan_r9rt", "ba_con", "escape_con"], 
    joint_retraining_path=[DATA_DIR/"Joint_Retraining_MS_Matched_Early_Stopping", DATA_DIR/"Joint_Retraining_MS_Matched_Early_Stopping_Updated"]
)


yd:
  Joint_Retraining_Early_Stopping/yd/holec_test: (160000, 18)
  Joint_Retraining_Early_Stopping/yd/holec_val: (72300, 17)
  Joint_Retraining_Early_Stopping/yd/ms_test: (3214300, 18)
  Joint_Retraining_Early_Stopping/yd/ms_val: (2075300, 17)
  not found: /home/pchati/PepCL_Manuscript/Data/Joint_Retraining_Early_Stopping_Updated/yd

episcan_r9:
  not found: /home/pchati/PepCL_Manuscript/Data/Joint_Retraining_Early_Stopping/episcan_r9
  Joint_Retraining_Early_Stopping_Updated/episcan_r9/cov_test: (326579, 18)
  Joint_Retraining_Early_Stopping_Updated/episcan_r9/episcan_r9_rt_test: (80000, 19)
  Joint_Retraining_Early_Stopping_Updated/episcan_r9/episcan_r9_rt_val: (33800, 18)
  Joint_Retraining_Early_Stopping_Updated/episcan_r9/ms_test: (3214300, 18)
  Joint_Retraining_Early_Stopping_Updated/episcan_r9/ms_val: (2075300, 17)

episcan_r9rt:
  not found: /home/pchati/PepCL_Manuscript/Data/Joint_Retraining_Early_Stopping/episcan_r9rt
  Joint_Retraining_Early_Stopping_Updated/episcan_r9rt/

#### YD experiments

In [ ]:
%%time
# load multiple dfs
path_prefix="YD"

results_dirs=[
    f"{DATA_DIR}/Lambda_Variations_w_Test_5_Seeds/{path_prefix}_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_Full_5_Seeds_w_Test/", 
]

df_names=["ms_val", "holec_val", "ms_test", "holec_test"]


loaded_dfs_all=load_and_merge_dir_dfs(results_dirs, df_names=df_names, key_cols=("seq", "allele", "label"), how="outer")

ms_val_for_holec=loaded_dfs_all["ms_val"]
holec_val=loaded_dfs_all["holec_val"]

ms_test_for_holec=loaded_dfs_all["ms_test"]
holec_test=loaded_dfs_all["holec_test"]

# match the ms val to the assay val
ms_val_for_holec_matched, holec_val=filter_by_allele_len_pairs(ms_val_for_holec, holec_val, allele_col="allele", seq_col="seq", filter_both=False)

ms_test_for_holec_matched, holec_test=filter_by_allele_len_pairs(ms_test_for_holec, holec_test, allele_col="allele", seq_col="seq", filter_both=False)

# create dfs dict
holec_dfs={
    "ms_val_matched": ms_val_for_holec_matched,
    "assay_val": holec_val,
    "ms_test_matched": ms_test_for_holec_matched,
    "assay_test": holec_test,
}

for df_name, df in holec_dfs.items():
    print(df_name)
    print_data_stats(df)
    print(end="\n")

[/home/pchati/PepCL_Manuscript/Data/Lambda_Variations_w_Test_5_Seeds/YD_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_Full_5_Seeds_w_Test/] found 4 df(s) (filtered to 4 requested)


dir 0: YD_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_Full_5_Seeds_w_Test: 100%|██████████| 4/4 [00:01<00:00,  2.44it/s]


total unique dfs in store: 4
ms_val_matched
Total: 80,000
Pos: 800 | 1.00%
Neg: 79,200 | 99.00%
N alleles: 16

assay_val
Total: 72,300
Pos: 723 | 1.00%
Neg: 71,577 | 99.00%
N alleles: 16

ms_test_matched
Total: 158,400
Pos: 1,584 | 1.00%
Neg: 156,816 | 99.00%
N alleles: 16

assay_test
Total: 160,000
Pos: 1,600 | 1.00%
Neg: 158,400 | 99.00%
N alleles: 16

CPU times: user 2.33 s, sys: 220 ms, total: 2.55 s
Wall time: 2.54 s


In [ ]:
# holec ap matrices
holec_ap_dfs={}

for df_name, df in tqdm(holec_dfs.items()):
    score_cols=["mhcprime"] + score_cols_per_dataset["holec"]
    ap_df=compute_ap_matrix_optimized(df, score_cols, label_col='label', allele_col='allele', disable_tqdm=True)
    holec_ap_dfs[df_name]=ap_df

  0%|          | 0/4 [00:00<?, ?it/s]

100%|██████████| 4/4 [00:02<00:00,  1.47it/s]


#### EpiScan experiments

 - Forgot to keep `dataset_source` in R9/RT, we can just add back after (simple fix). For future runs, the script has already been corrected.

##### R9

In [ ]:
%%time
# load multiple dfs
path_prefix="EpiScan_R9"

results_dirs=[
    f"{DATA_DIR}/Lambda_Variations_w_Test_5_Seeds/{path_prefix}_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_Full_5_Seeds_w_Test/", 
]


df_names=["ms_val", "episcan_r9_rt_val", "ms_test", "episcan_r9_rt_test", "cov_test"]

loaded_dfs_all=load_and_merge_dir_dfs(results_dirs, df_names=df_names, key_cols=("seq", "allele", "label"), how="outer")

ms_val_for_episcan_r9=loaded_dfs_all["ms_val"]
episcan_r9_rt_val_for_r9=loaded_dfs_all["episcan_r9_rt_val"]
ms_test_for_episcan_r9=loaded_dfs_all["ms_test"]
episcan_r9_rt_test_for_r9=loaded_dfs_all["episcan_r9_rt_test"]
cov_test_for_r9=loaded_dfs_all["cov_test"]

# match the ms val to the assay val
ms_val_for_episcan_r9_matched, episcan_r9_rt_val_for_r9=filter_by_allele_len_pairs(ms_val_for_episcan_r9, episcan_r9_rt_val_for_r9, allele_col="allele", seq_col="seq", filter_both=False)

ms_test_for_episcan_r9_matched, episcan_r9_rt_test_for_r9=filter_by_allele_len_pairs(ms_test_for_episcan_r9, episcan_r9_rt_test_for_r9, allele_col="allele", seq_col="seq", filter_both=False)

# create dfs dict
episcan_r9_dfs={
    "ms_val_matched": ms_val_for_episcan_r9_matched,
    "assay_val": episcan_r9_rt_val_for_r9,
    # "r9_val": episcan_r9_rt_val_for_r9.query("dataset_source == 'r9'"),
    # "rt_val": episcan_r9_rt_val_for_r9.query("dataset_source == 'rt'"),
    "ms_test_matched": ms_test_for_episcan_r9_matched,
    "assay_test": episcan_r9_rt_test_for_r9,
    "cov_test": cov_test_for_r9,

}

[/home/pchati/PepCL_Manuscript/Data/Lambda_Variations_w_Test_5_Seeds/EpiScan_R9_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_Full_5_Seeds_w_Test/] found 5 df(s) (filtered to 5 requested)


dir 0: EpiScan_R9_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_Full_5_Seeds_w_Test:   0%|          | 0/5 [00:00<?, ?it/s]

dir 0: EpiScan_R9_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_Full_5_Seeds_w_Test: 100%|██████████| 5/5 [00:01<00:00,  4.17it/s]


total unique dfs in store: 5
CPU times: user 1.18 s, sys: 128 ms, total: 1.31 s
Wall time: 1.37 s


In [ ]:
# r9 aps
episcan_r9_ap_dfs={}

for df_name, df in tqdm(episcan_r9_dfs.items()):
    score_cols=["mhcprime"] + score_cols_per_dataset["episcan_r9"]
    ap_df = compute_ap_matrix_optimized(df, score_cols, label_col='label', allele_col='allele', disable_tqdm=True)
    episcan_r9_ap_dfs[df_name] = ap_df

n_iterations=100
allele_pos_counts = {"A0201": 100, "A0301": 100, "B0801": 100, "B5701": 100}

# per-allele AP matrix, pos-focused (negs from r9 only)
print("EpiScan R9")
episcan_r9_val_set_for_r9=episcan_r9_dfs["assay_val"].query("dataset_source == 'r9'")
episcan_rt_val_set_for_r9=episcan_r9_dfs["assay_val"].query("dataset_source == 'rt'")

score_cols=["mhcprime"] + score_cols_per_dataset["episcan_r9"]
score_cols=keep_present_cols(episcan_r9_dfs["assay_val"], score_cols)

r9rt_pos_r9_neg_per_allele_ap_for_r9 = mixed_df_per_allele_ap_pos_focused_fast(episcan_r9_val_set_for_r9, episcan_rt_val_set_for_r9.query("label == 1"), score_cols, allele_pos_counts, label_col1='label', label_col2='label', allele_col='allele', neg_ratio=99, n_iterations=n_iterations, seed=42, chunk_size=256, num_threads=16)

episcan_r9_ap_dfs["assay_val_mixture"]=r9rt_pos_r9_neg_per_allele_ap_for_r9

# for test
print("EpiScan R9")
episcan_r9_test_set_for_r9=episcan_r9_dfs["assay_test"].query("dataset_source == 'r9'")
episcan_rt_test_set_for_r9=episcan_r9_dfs["assay_test"].query("dataset_source == 'rt'")

score_cols=["mhcprime"] + score_cols_per_dataset["episcan_r9"]
score_cols=keep_present_cols(episcan_r9_dfs["assay_test"], score_cols)

r9rt_pos_r9_neg_per_allele_ap_for_r9_test = mixed_df_per_allele_ap_pos_focused_fast(episcan_r9_test_set_for_r9, episcan_rt_test_set_for_r9.query("label == 1"), score_cols, allele_pos_counts, label_col1='label', label_col2='label', allele_col='allele', neg_ratio=99, n_iterations=n_iterations, seed=42, chunk_size=256, num_threads=16)

episcan_r9_ap_dfs["assay_test_mixture"]=r9rt_pos_r9_neg_per_allele_ap_for_r9_test

  0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 5/5 [00:01<00:00,  3.00it/s]


EpiScan R9
[WARNING] Allele A0301 has only 83 positives (df1=33, df2=50), below requested 100
[WARNING] Allele B0801 has only 55 positives (df1=5, df2=50), below requested 100


Iterations: 100%|██████████| 100/100 [00:02<00:00, 44.74it/s]


EpiScan R9


Iterations: 100%|██████████| 100/100 [00:03<00:00, 31.41it/s]


##### R9/RT

In [ ]:
%%time
# load multiple dfs
path_prefix="EpiScan_R9_RT"


results_dirs=[
    f"{DATA_DIR}/Lambda_Variations_w_Test_5_Seeds/{path_prefix}_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_Full_5_Seeds_w_Test/", 
]

df_names=["ms_val", "episcan_r9_rt_val", "ms_test", "episcan_r9_rt_test", "cov_test"]

loaded_dfs_all=load_and_merge_dir_dfs(results_dirs, df_names=df_names, key_cols=("seq", "allele", "label"), how="outer")

ms_val_for_episcan_r9rt=loaded_dfs_all["ms_val"]
episcan_r9_rt_val_for_r9rt=loaded_dfs_all["episcan_r9_rt_val"]
ms_test_for_episcan_r9rt=loaded_dfs_all["ms_test"]
episcan_r9_rt_test_for_r9rt=loaded_dfs_all["episcan_r9_rt_test"]
cov_test_for_r9rt=loaded_dfs_all["cov_test"]

# match the ms val to the assay val
ms_val_for_episcan_r9rt_matched, episcan_r9_rt_val_for_r9rt=filter_by_allele_len_pairs(ms_val_for_episcan_r9rt, episcan_r9_rt_val_for_r9rt, allele_col="allele", seq_col="seq", filter_both=False)
ms_test_for_episcan_r9rt_matched, episcan_r9_rt_test_for_r9rt=filter_by_allele_len_pairs(ms_test_for_episcan_r9rt, episcan_r9_rt_test_for_r9rt, allele_col="allele", seq_col="seq", filter_both=False)

# create dfs dict
episcan_r9rt_dfs={
    "ms_val_matched": ms_val_for_episcan_r9rt_matched,
    "assay_val": episcan_r9_rt_val_for_r9rt,
    # "r9_val": episcan_r9_rt_val_for_r9rt.query("dataset_source == 'r9'"),
    # "rt_val": episcan_r9_rt_val_for_r9rt.query("dataset_source == 'rt'"),
    "ms_test_matched": ms_test_for_episcan_r9rt_matched,
    "assay_test": episcan_r9_rt_test_for_r9rt,
    "cov_test": cov_test_for_r9rt,
}

for df_name, df in episcan_r9rt_dfs.items():
    print(df_name)
    print_data_stats(df)
    print(end="\n")

[/home/pchati/PepCL_Manuscript/Data/Lambda_Variations_w_Test_5_Seeds/EpiScan_R9_RT_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_Full_5_Seeds_w_Test/] found 5 df(s) (filtered to 5 requested)


dir 0: EpiScan_R9_RT_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_Full_5_Seeds_w_Test: 100%|██████████| 5/5 [00:01<00:00,  2.91it/s]


total unique dfs in store: 5
ms_val_matched
Total: 20,000
Pos: 200 | 1.00%
Neg: 19,800 | 99.00%
N alleles: 4

assay_val
Total: 33,800
Pos: 338 | 1.00%
Neg: 33,462 | 99.00%
N alleles: 4

ms_test_matched
Total: 40,000
Pos: 400 | 1.00%
Neg: 39,600 | 99.00%
N alleles: 4

assay_test
Total: 80,000
Pos: 800 | 1.00%
Neg: 79,200 | 99.00%
N alleles: 4

cov_test
Total: 326,579
Pos: 767 | 0.23%
Neg: 325,812 | 99.77%
N alleles: 11

CPU times: user 1.99 s, sys: 155 ms, total: 2.14 s
Wall time: 2.13 s


In [ ]:
# r9rt aps
episcan_r9rt_ap_dfs={}

for df_name, df in tqdm(episcan_r9rt_dfs.items()):
    score_cols=["mhcprime"] + score_cols_per_dataset["episcan_r9rt"]
    ap_df = compute_ap_matrix_optimized(df, score_cols, label_col='label', allele_col='allele', disable_tqdm=True)
    episcan_r9rt_ap_dfs[df_name] = ap_df


n_iterations=100
allele_pos_counts = {"A0201": 100, "A0301": 100, "B0801": 100, "B5701": 100}

# per-allele AP matrix, both (negs split across r9 and rt)
print("EpiScan R9/RT")
episcan_r9_val_set_for_r9rt=episcan_r9rt_dfs["assay_val"].query("dataset_source == 'r9'")
episcan_rt_val_set_for_r9rt=episcan_r9rt_dfs["assay_val"].query("dataset_source == 'rt'")

score_cols=["mhcprime"] + score_cols_per_dataset["episcan_r9rt"]
score_cols=keep_present_cols(episcan_r9rt_dfs["assay_val"], score_cols)

r9rt_pos_r9rt_neg_per_allele_ap_for_r9rt = mixed_df_per_allele_ap_balanced_fast(episcan_r9_val_set_for_r9rt, episcan_rt_val_set_for_r9rt, score_cols, label_col1='label', label_col2='label', allele_col='allele', neg_ratio=99, n_iterations=n_iterations, seed=42, chunk_size=256, num_threads=16)

episcan_r9rt_ap_dfs["assay_val_mixture"]=r9rt_pos_r9rt_neg_per_allele_ap_for_r9rt

# for test
print("EpiScan R9/RT")
episcan_r9_test_set_for_r9rt=episcan_r9rt_dfs["assay_test"].query("dataset_source == 'r9'")
episcan_rt_test_set_for_r9rt=episcan_r9rt_dfs["assay_test"].query("dataset_source == 'rt'")

score_cols=["mhcprime"] + score_cols_per_dataset["episcan_r9rt"]
score_cols=keep_present_cols(episcan_r9rt_dfs["assay_test"], score_cols)

r9rt_pos_r9rt_neg_per_allele_ap_for_r9rt_test = mixed_df_per_allele_ap_both_fast(episcan_r9_test_set_for_r9rt, episcan_rt_test_set_for_r9rt, score_cols, allele_pos_counts, label_col1='label', label_col2='label', allele_col='allele', neg_ratio=99, n_iterations=n_iterations, seed=42, chunk_size=256, num_threads=16)

episcan_r9rt_ap_dfs["assay_test_mixture"]=r9rt_pos_r9rt_neg_per_allele_ap_for_r9rt_test

100%|██████████| 5/5 [00:02<00:00,  1.76it/s]


EpiScan R9/RT


Iterations: 100%|██████████| 100/100 [00:02<00:00, 45.46it/s]


EpiScan R9/RT


Iterations: 100%|██████████| 100/100 [00:03<00:00, 25.53it/s]


#### BA experiments

In [ ]:
%%time
# load multiple dfs
path_prefix="BA_CON"


results_dirs=[
    f"{DATA_DIR}/Lambda_Variations_w_Test_5_Seeds/{path_prefix}_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_Full_5_Seeds_w_Test/", 
]

df_names=["ms_val", "ba_val", "ms_test", "ba_test"]

loaded_dfs_all=load_and_merge_dir_dfs(results_dirs, df_names=df_names, key_cols=("seq", "allele", "label"), how="outer")

ms_val_for_ba_con=loaded_dfs_all["ms_val"]
ba_con_val=loaded_dfs_all["ba_val"]
ms_test_for_ba_con=loaded_dfs_all["ms_test"]
ba_con_test=loaded_dfs_all["ba_test"]

# match the ms val to the assay val
ms_val_for_ba_con_matched, ba_con_val=filter_by_allele_len_pairs(ms_val_for_ba_con, ba_con_val, allele_col="allele", seq_col="seq", filter_both=False)
ms_test_for_ba_con_matched, ba_con_test=filter_by_allele_len_pairs(ms_test_for_ba_con, ba_con_test, allele_col="allele", seq_col="seq", filter_both=False)

# create dfs dict
ba_con_dfs={
    "ms_val_matched": ms_val_for_ba_con_matched,
    "assay_val": ba_con_val,
    "ms_test_matched": ms_test_for_ba_con_matched,
    "assay_test": ba_con_test,
}

for df_name, df in ba_con_dfs.items():
    print(df_name)
    print_data_stats(df)
    print(end="\n")

[/home/pchati/PepCL_Manuscript/Data/Lambda_Variations_w_Test_5_Seeds/BA_CON_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_Full_5_Seeds_w_Test/] found 4 df(s) (filtered to 4 requested)


dir 0: BA_CON_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_Full_5_Seeds_w_Test: 100%|██████████| 4/4 [00:06<00:00,  1.54s/it]


total unique dfs in store: 4
ms_val_matched
Total: 327,500
Pos: 3,275 | 1.00%
Neg: 324,225 | 99.00%
N alleles: 41

assay_val
Total: 3,052
Pos: 1,526 | 50.00%
Neg: 1,526 | 50.00%
N alleles: 41

ms_test_matched
Total: 1,361,400
Pos: 13,614 | 1.00%
Neg: 1,347,786 | 99.00%
N alleles: 51

assay_test
Total: 4,221
Pos: 2,077 | 49.21%
Neg: 2,144 | 50.79%
N alleles: 51

CPU times: user 8.93 s, sys: 1.52 s, total: 10.4 s
Wall time: 10.4 s


In [ ]:
%%time
# ba aps
ba_con_ap_dfs={}

for df_name, df in tqdm(ba_con_dfs.items()):
    score_cols=["mhcprime"] + score_cols_per_dataset["ba_con"]
    if df_name == "ms_val" or df_name == "ms_val_matched" or df_name == "ms_test" or df_name == "ms_test_matched":
        print(f"{df_name}: AP branch")
        ap_df = compute_ap_matrix_optimized(df, score_cols, label_col='label', allele_col='allele', disable_tqdm=True)
        ba_con_ap_dfs[df_name] = ap_df
    else:
        print(f"{df_name}: SPEARMAN branch")
        spearman_df = compute_spearman_matrix_optimized(df, score_cols, reference_col="ba_value", allele_col='allele', disable_tqdm=True)
        ba_con_ap_dfs[df_name] = spearman_df

        df["all"]=1
        spearman_df_all = compute_spearman_matrix_optimized(df, score_cols, reference_col="ba_value", allele_col='all', disable_tqdm=True)
        ba_con_ap_dfs["assay_val_all"] = spearman_df_all

        spearman_df_pos = compute_spearman_matrix_optimized(df.query("label == 1"), score_cols, reference_col="ba_value", allele_col='allele', disable_tqdm=True)
        ba_con_ap_dfs["assay_val_pos"] = spearman_df_pos

        spearman_df_all_pos = compute_spearman_matrix_optimized(df.query("label == 1"), score_cols, reference_col="ba_value", allele_col='all', disable_tqdm=True)
        ba_con_ap_dfs["assay_val_all_pos"] = spearman_df_all_pos

  0%|          | 0/4 [00:00<?, ?it/s]

ms_val_matched: AP branch


 25%|██▌       | 1/4 [00:02<00:07,  2.38s/it]

assay_val: SPEARMAN branch


 50%|█████     | 2/4 [00:03<00:03,  1.76s/it]

ms_test_matched: AP branch


 75%|███████▌  | 3/4 [00:08<00:02,  2.99s/it]

assay_test: SPEARMAN branch


100%|██████████| 4/4 [00:09<00:00,  2.45s/it]

CPU times: user 9.6 s, sys: 242 ms, total: 9.84 s
Wall time: 9.81 s


#### ESCAPE-seq experiments

In [ ]:
%%time
# load multiple dfs
path_prefix="ESCAPE_seq_CON"


results_dirs=[
    f"{DATA_DIR}/Lambda_Variations_w_Test_5_Seeds/{path_prefix}_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_Full_5_Seeds_w_Test/",
]

df_names=["ms_val", "escape_val", "ms_test", "escape_test"]

loaded_dfs_all=load_and_merge_dir_dfs(results_dirs, df_names=df_names, key_cols=("seq", "allele", "label"), how="outer")

ms_val_for_escape_con=loaded_dfs_all["ms_val"]
escape_con_val=loaded_dfs_all["escape_val"]
ms_test_for_escape_con=loaded_dfs_all["ms_test"]
escape_con_test=loaded_dfs_all["escape_test"]

# match the ms val to the assay val
ms_val_for_escape_con_matched, escape_con_val=filter_by_allele_len_pairs(ms_val_for_escape_con, escape_con_val, allele_col="allele", seq_col="seq", filter_both=False)
ms_test_for_escape_con_matched, escape_con_test=filter_by_allele_len_pairs(ms_test_for_escape_con, escape_con_test, allele_col="allele", seq_col="seq", filter_both=False)

# create dfs dict
escape_con_dfs={
    # "ms_val": ms_val_for_escape_con,
    "ms_val_matched": ms_val_for_escape_con_matched,
    "assay_val": escape_con_val,
    "ms_test_matched": ms_test_for_escape_con_matched,
    "assay_test": escape_con_test,
}

for df_name, df in escape_con_dfs.items():
    print(df_name)
    print_data_stats(df)
    print(end="\n")

[/home/pchati/PepCL_Manuscript/Data/Lambda_Variations_w_Test_5_Seeds/ESCAPE_seq_CON_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_Full_5_Seeds_w_Test/] found 4 df(s) (filtered to 4 requested)


dir 0: ESCAPE_seq_CON_Lambda_Ablations_w_Alt_MHCPrime_Default_Params_Full_5_Seeds_w_Test: 100%|██████████| 4/4 [00:01<00:00,  3.10it/s]


total unique dfs in store: 4
ms_val_matched
Total: 185,000
Pos: 1,850 | 1.00%
Neg: 183,150 | 99.00%
N alleles: 37

assay_val
Total: 1,315
Pos: 657 | 49.96%
Neg: 658 | 50.04%
N alleles: 41

ms_test_matched
Total: 361,900
Pos: 3,619 | 1.00%
Neg: 358,281 | 99.00%
N alleles: 37

assay_test
Total: 1,446
Pos: 718 | 49.65%
Neg: 728 | 50.35%
N alleles: 41

CPU times: user 1.9 s, sys: 276 ms, total: 2.18 s
Wall time: 2.17 s


In [ ]:
%%time
# escape aps
escape_con_ap_dfs={}

for df_name, df in tqdm(escape_con_dfs.items()):
    score_cols=["mhcprime"] + score_cols_per_dataset["escape_con"]
    if df_name == "ms_val" or df_name == "ms_val_matched" or df_name == "ms_test" or df_name == "ms_test_matched":
        print(f"{df_name}: AP branch")
        ap_df = compute_ap_matrix_optimized(df, score_cols, label_col='label', allele_col='allele', disable_tqdm=True)
        escape_con_ap_dfs[df_name] = ap_df
    else:
        print(f"{df_name}: SPEARMAN branch")
        spearman_df = compute_spearman_matrix_optimized(df, score_cols, reference_col="escore", allele_col='allele', disable_tqdm=True)
        escape_con_ap_dfs[df_name] = spearman_df

        df["all"]=1
        spearman_df_all = compute_spearman_matrix_optimized(df, score_cols, reference_col="escore", allele_col='all', disable_tqdm=True)
        escape_con_ap_dfs["assay_val_all"] = spearman_df_all

        spearman_df_pos = compute_spearman_matrix_optimized(df.query("label == 1"), score_cols, reference_col="escore", allele_col='allele', disable_tqdm=True)
        escape_con_ap_dfs["assay_val_pos"] = spearman_df_pos

        spearman_df_all_pos = compute_spearman_matrix_optimized(df.query("label == 1"), score_cols, reference_col="escore", allele_col='all', disable_tqdm=True)
        escape_con_ap_dfs["assay_val_all_pos"] = spearman_df_all_pos


  0%|          | 0/4 [00:00<?, ?it/s]

ms_val_matched: AP branch


 25%|██▌       | 1/4 [00:00<00:02,  1.17it/s]

assay_val: SPEARMAN branch


 50%|█████     | 2/4 [00:02<00:02,  1.12s/it]

ms_test_matched: AP branch


 75%|███████▌  | 3/4 [00:03<00:01,  1.23s/it]

assay_test: SPEARMAN branch


100%|██████████| 4/4 [00:04<00:00,  1.20s/it]

CPU times: user 4.82 s, sys: 8.71 ms, total: 4.83 s
Wall time: 4.82 s


#### Component performances for validation and test

In [ ]:
# focus on seeds 0 to 4, as this is comparable then to the train used for retraining
# get the component ablations from these

focus_seeds=[0,1,2,3,4]

verbose=False

holec_rep_df, holec_summary_df_val, contrasts_df, factorial_df=analyze_pepcl_components(
    holec_ap_dfs["ms_val_matched"],
    holec_ap_dfs["assay_val"],
    score_cols_per_dataset["holec"],
    method_prefixes_per_dataset["holec"],
    dr_lambda=1e-5,
    verbose=verbose,
    focus_seeds=focus_seeds,
    ref_col="mhcprime",
)

episcan_r9_rep_df, episcan_r9_summary_df_val, contrasts_df, factorial_df=analyze_pepcl_components(
    episcan_r9_ap_dfs["ms_val_matched"],
    episcan_r9_ap_dfs["assay_val_mixture"],
    score_cols_per_dataset["episcan_r9"],
    method_prefixes_per_dataset["episcan_r9"],
    dr_lambda=1e-5,
    verbose=verbose,
    focus_seeds=focus_seeds,
    ref_col="mhcprime",
)

episcan_r9rt_rep_df, episcan_r9rt_summary_df_val, contrasts_df, factorial_df=analyze_pepcl_components(
    episcan_r9rt_ap_dfs["ms_val_matched"],
    episcan_r9rt_ap_dfs["assay_val_mixture"],
    score_cols_per_dataset["episcan_r9rt"],
    method_prefixes_per_dataset["episcan_r9rt"],
    dr_lambda=1e-5,
    verbose=verbose,
    focus_seeds=focus_seeds,
    ref_col="mhcprime",
)


ba_con_rep_df, ba_con_summary_df_val, contrasts_df, factorial_df=analyze_pepcl_components(
    ba_con_ap_dfs["ms_val_matched"],
    ba_con_ap_dfs["assay_val"],
    score_cols_per_dataset["ba_con"],
    method_prefixes_per_dataset["ba_con"],
    dr_lambda=1e-5,
    verbose=verbose,
    focus_seeds=focus_seeds,
    ref_col="mhcprime",
)

escape_con_rep_df, escape_con_summary_df_val, contrasts_df, factorial_df=analyze_pepcl_components(
    escape_con_ap_dfs["ms_val_matched"],
    escape_con_ap_dfs["assay_val"],
    score_cols_per_dataset["escape_con"],
    method_prefixes_per_dataset["escape_con"],
    dr_lambda=1e-5,
    verbose=verbose,
    focus_seeds=focus_seeds,
    ref_col="mhcprime",
)
# for joint retraining optimal point
summary_dfs_val={
    "yd": holec_summary_df_val,
    "episcan_r9": episcan_r9_summary_df_val,
    "episcan_r9rt": episcan_r9rt_summary_df_val,
    "ba_con": ba_con_summary_df_val,
    "escape_con": escape_con_summary_df_val,
}


In [ ]:
# focus on seeds 0 to 4, as this is comparable then to the train used for retraining
# get the component ablations from these

focus_seeds=[0,1,2,3,4]

verbose=False

holec_rep_df, holec_summary_df_test, contrasts_df, factorial_df=analyze_pepcl_components(
    holec_ap_dfs["ms_test_matched"],
    holec_ap_dfs["assay_test"],
    score_cols_per_dataset["holec"],
    method_prefixes_per_dataset["holec"],
    dr_lambda=1e-5,
    verbose=verbose,
    focus_seeds=focus_seeds,
    ref_col="mhcprime",
)

episcan_r9_rep_df, episcan_r9_summary_df_test, contrasts_df, factorial_df=analyze_pepcl_components(
    episcan_r9_ap_dfs["ms_test_matched"],
    episcan_r9_ap_dfs["assay_test_mixture"],
    score_cols_per_dataset["episcan_r9"],
    method_prefixes_per_dataset["episcan_r9"],
    dr_lambda=1e-5,
    verbose=verbose,
    focus_seeds=focus_seeds,
    ref_col="mhcprime",
)

episcan_r9rt_rep_df, episcan_r9rt_summary_df_test, contrasts_df, factorial_df=analyze_pepcl_components(
    episcan_r9rt_ap_dfs["ms_test_matched"],
    episcan_r9rt_ap_dfs["assay_test_mixture"],
    score_cols_per_dataset["episcan_r9rt"],
    method_prefixes_per_dataset["episcan_r9rt"],
    dr_lambda=1e-5,
    verbose=verbose,
    focus_seeds=focus_seeds,
    ref_col="mhcprime",
)


ba_con_rep_df, ba_con_summary_df_test, contrasts_df, factorial_df=analyze_pepcl_components(
    ba_con_ap_dfs["ms_test_matched"],
    ba_con_ap_dfs["assay_test"],
    score_cols_per_dataset["ba_con"],
    method_prefixes_per_dataset["ba_con"],
    dr_lambda=1e-5,
    verbose=verbose,
    focus_seeds=focus_seeds,
    ref_col="mhcprime",
)

escape_con_rep_df, escape_con_summary_df_test, contrasts_df, factorial_df=analyze_pepcl_components(
    escape_con_ap_dfs["ms_test_matched"],
    escape_con_ap_dfs["assay_test"],
    score_cols_per_dataset["escape_con"],
    method_prefixes_per_dataset["escape_con"],
    dr_lambda=1e-5,
    verbose=verbose,
    focus_seeds=focus_seeds,
    ref_col="mhcprime",
)
# for joint retraining optimal point
summary_dfs_test={
    "yd": holec_summary_df_test,
    "episcan_r9": episcan_r9_summary_df_test,
    "episcan_r9rt": episcan_r9rt_summary_df_test,
    "ba_con": ba_con_summary_df_test,
    "escape_con": escape_con_summary_df_test,
}


In [ ]:
# save steps
save_every_n_steps=steps_to_max(15720, interval=100, start=1)

In [ ]:
# init dicts (full ms)
assay_ms_val_full_ms_dfs={}
assay_ms_val_full_ms_ap_dfs={}

In [ ]:
# init dicts (matched ms)
assay_ms_val_matched_ms_dfs={}
assay_ms_val_matched_ms_ap_dfs={}

In [ ]:
%%time
# get performance for holec val across steps (for full ms)
score_cols=joint_retraining_score_cols("yd_retraining_steps_15720_seed_42", save_every_n_steps, 15720, include_final=True, return_map=False)
# score_cols+=["yd_retraining_steps_15720_seed_42"]

assay_val=joint_data['yd']['holec_val']
assay_alleles=assay_val.allele.unique().tolist()
ms_val=joint_data['yd']['ms_val']

ms_val["seq_len"]=ms_val.seq.str.replace("$", "").apply(len)
ms_val_matched, assay_val=filter_by_allele_len_pairs(ms_val, assay_val, allele_col="allele", seq_col="seq", filter_both=False)

ms_ap_df=compute_ap_matrix_optimized(ms_val_matched, score_cols, label_col='label', allele_col='allele', disable_tqdm=False)

assay_ap_df=compute_ap_matrix_optimized(assay_val, score_cols, label_col='label', allele_col='allele', disable_tqdm=False)

assay_ms_val_full_ms_dfs["yd"]={"ms_val": ms_val, "ms_val_matched": ms_val_matched, "assay_val": assay_val}

assay_ms_val_full_ms_ap_dfs["yd"]={"ms_val_matched": ms_ap_df, "assay_val": assay_ap_df}

100%|██████████| 16/16 [00:02<00:00,  6.08it/s]

CPU times: user 8.63 s, sys: 2.9 s, total: 11.5 s
Wall time: 11.5 s


In [ ]:
%%time
# get performance for holec val across steps (for matched ms)
score_cols=joint_retraining_score_cols("yd_retraining_ms_matched_steps_15720_seed_42", save_every_n_steps, 15720, include_final=True, return_map=False)

assay_val=joint_data_ms_matched['yd']['holec_val']
assay_alleles=assay_val.allele.unique().tolist()
ms_val=joint_data_ms_matched['yd']['ms_val']

ms_val["seq_len"]=ms_val.seq.str.replace("$", "").apply(len)
ms_val_matched, assay_val=filter_by_allele_len_pairs(ms_val, assay_val, allele_col="allele", seq_col="seq", filter_both=False)

ms_ap_df=compute_ap_matrix_optimized(ms_val_matched, score_cols, label_col='label', allele_col='allele', disable_tqdm=False)

assay_ap_df=compute_ap_matrix_optimized(assay_val, score_cols, label_col='label', allele_col='allele', disable_tqdm=False)

assay_ms_val_matched_ms_dfs["yd"]={"ms_val": ms_val, "ms_val_matched": ms_val_matched, "assay_val": assay_val}

assay_ms_val_matched_ms_ap_dfs["yd"]={"ms_val_matched": ms_ap_df, "assay_val": assay_ap_df}

100%|██████████| 16/16 [00:02<00:00,  6.19it/s]

CPU times: user 7.8 s, sys: 1.8 s, total: 9.6 s
Wall time: 9.57 s


In [ ]:
%%time
# get performance for episcan r9 val across steps (for full ms)

assay_val=joint_data["episcan_r9"]["episcan_r9_rt_val"]
ms_val=joint_data["episcan_r9"]["ms_val"]

ms_val["seq_len"]=ms_val.seq.str.replace("$", "").apply(len)
ms_val_matched, assay_val=filter_by_allele_len_pairs(ms_val, assay_val, allele_col="allele", seq_col="seq", filter_both=False)

score_cols=joint_retraining_score_cols("episcan_r9_retraining_steps_15720_seed_42", save_every_n_steps, 15720, include_final=True, return_map=False)

ms_ap_df=compute_ap_matrix_optimized(ms_val_matched, score_cols, label_col='label', allele_col='allele', disable_tqdm=False)

n_iterations=100
allele_pos_counts = {"A0201": 100, "A0301": 100, "B0801": 100, "B5701": 100}

# per-allele AP matrix, pos-focused (negs from r9 only)
print("EpiScan R9")
episcan_r9_val_set_for_r9=assay_val.query("dataset_source == 'r9'")
episcan_rt_val_set_for_r9=assay_val.query("dataset_source == 'rt'")

score_cols=keep_present_cols(episcan_r9_val_set_for_r9, score_cols)

r9rt_pos_r9_neg_per_allele_ap_for_r9 = mixed_df_per_allele_ap_pos_focused_fast(episcan_r9_val_set_for_r9, episcan_rt_val_set_for_r9.query("label == 1"), score_cols, allele_pos_counts, label_col1='label', label_col2='label', allele_col='allele', neg_ratio=99, n_iterations=n_iterations, seed=42, chunk_size=256, num_threads=16)

assay_val_mixture=pd.concat([episcan_r9_val_set_for_r9, episcan_rt_val_set_for_r9.query("label == 1")])

assay_ms_val_full_ms_dfs["episcan_r9"]={"ms_val": ms_val, "ms_val_matched": ms_val_matched, "assay_val": assay_val_mixture} # use proper assay val mixture

assay_ms_val_full_ms_ap_dfs["episcan_r9"]={"ms_val_matched": ms_ap_df, "assay_val": r9rt_pos_r9_neg_per_allele_ap_for_r9}

100%|██████████| 4/4 [00:00<00:00,  5.73it/s]


EpiScan R9
[WARNING] Allele A0301 has only 83 positives (df1=33, df2=50), below requested 100
[WARNING] Allele B0801 has only 55 positives (df1=5, df2=50), below requested 100


Iterations: 100%|██████████| 100/100 [00:07<00:00, 14.14it/s]

CPU times: user 1min 9s, sys: 2.3 s, total: 1min 11s
Wall time: 12.1 s


In [ ]:
%%time
# get performance for episcan r9 val across steps (for matched ms)

assay_val=joint_data_ms_matched["episcan_r9"]["episcan_r9_rt_val"]
ms_val=joint_data_ms_matched["episcan_r9"]["ms_val"]

ms_val["seq_len"]=ms_val.seq.str.replace("$", "").apply(len)
ms_val_matched, assay_val=filter_by_allele_len_pairs(ms_val, assay_val, allele_col="allele", seq_col="seq", filter_both=False)

score_cols=joint_retraining_score_cols("episcan_r9_retraining_ms_matched_steps_15720_seed_42", save_every_n_steps, 15720, include_final=True, return_map=False)

ms_ap_df=compute_ap_matrix_optimized(ms_val_matched, score_cols, label_col='label', allele_col='allele', disable_tqdm=False)

n_iterations=100
allele_pos_counts = {"A0201": 100, "A0301": 100, "B0801": 100, "B5701": 100}

# per-allele AP matrix, pos-focused (negs from r9 only)
print("EpiScan R9")
episcan_r9_val_set_for_r9=assay_val.query("dataset_source == 'r9'")
episcan_rt_val_set_for_r9=assay_val.query("dataset_source == 'rt'")

score_cols=keep_present_cols(episcan_r9_val_set_for_r9, score_cols)

r9rt_pos_r9_neg_per_allele_ap_for_r9 = mixed_df_per_allele_ap_pos_focused_fast(episcan_r9_val_set_for_r9, episcan_rt_val_set_for_r9.query("label == 1"), score_cols, allele_pos_counts, label_col1='label', label_col2='label', allele_col='allele', neg_ratio=99, n_iterations=n_iterations, seed=42, chunk_size=256, num_threads=16)

assay_val_mixture=pd.concat([episcan_r9_val_set_for_r9, episcan_rt_val_set_for_r9.query("label == 1")])

assay_ms_val_matched_ms_dfs["episcan_r9"]={"ms_val": ms_val, "ms_val_matched": ms_val_matched, "assay_val": assay_val_mixture} # use proper assay val mixture

assay_ms_val_matched_ms_ap_dfs["episcan_r9"]={"ms_val_matched": ms_ap_df, "assay_val": r9rt_pos_r9_neg_per_allele_ap_for_r9}

100%|██████████| 4/4 [00:01<00:00,  3.36it/s]


EpiScan R9
[WARNING] Allele A0301 has only 83 positives (df1=33, df2=50), below requested 100
[WARNING] Allele B0801 has only 55 positives (df1=5, df2=50), below requested 100


Iterations: 100%|██████████| 100/100 [00:06<00:00, 15.00it/s]

CPU times: user 1min 4s, sys: 1.71 s, total: 1min 6s
Wall time: 13.6 s


In [ ]:
%%time
# get performance for episcan r9/rt val across steps (for full ms)

assay_val=joint_data["episcan_r9rt"]["episcan_r9_rt_val"]
ms_val=joint_data["episcan_r9rt"]["ms_val"]

assay_alleles=assay_val.allele.unique().tolist()

score_cols=joint_retraining_score_cols("episcan_r9rt_retraining_steps_15720_seed_42", save_every_n_steps, 15720, include_final=True, return_map=False)

ms_val["seq_len"]=ms_val.seq.str.replace("$", "").apply(len)
ms_val_matched, assay_val=filter_by_allele_len_pairs(ms_val, assay_val, allele_col="allele", seq_col="seq", filter_both=False)

print("EpiScan R9/RT")
episcan_r9_val_set_for_r9rt=assay_val.query("dataset_source == 'r9'")
episcan_rt_val_set_for_r9rt=assay_val.query("dataset_source == 'rt'")

ms_ap_df=compute_ap_matrix_optimized(ms_val_matched, score_cols, label_col='label', allele_col='allele', disable_tqdm=False)

score_cols=keep_present_cols(episcan_r9_val_set_for_r9rt, score_cols)

n_iterations=100
r9rt_pos_r9rt_neg_per_allele_ap_for_r9rt = mixed_df_per_allele_ap_balanced_fast(episcan_r9_val_set_for_r9rt, episcan_rt_val_set_for_r9rt, score_cols, label_col1='label', label_col2='label', allele_col='allele', neg_ratio=99, n_iterations=n_iterations, seed=42, chunk_size=256, num_threads=16)

assay_ms_val_full_ms_dfs["episcan_r9rt"]={"ms_val": ms_val, "ms_val_matched": ms_val_matched, "assay_val": assay_val}

assay_ms_val_full_ms_ap_dfs["episcan_r9rt"]={"ms_val_matched": ms_ap_df, "assay_val": r9rt_pos_r9rt_neg_per_allele_ap_for_r9rt}

EpiScan R9/RT


Iterations: 100%|██████████| 100/100 [00:07<00:00, 13.63it/s]

CPU times: user 1min 26s, sys: 2.45 s, total: 1min 28s
Wall time: 14.7 s


In [ ]:
%%time
# get performance for episcan r9/rt val across steps (for matched ms)

assay_val=joint_data_ms_matched["episcan_r9rt"]["episcan_r9_rt_val"]
ms_val=joint_data_ms_matched["episcan_r9rt"]["ms_val"]

score_cols=joint_retraining_score_cols("episcan_r9rt_retraining_ms_matched_steps_15720_seed_42", save_every_n_steps, 15720, include_final=True, return_map=False)

assay_alleles=assay_val.allele.unique().tolist()

ms_val["seq_len"]=ms_val.seq.str.replace("$", "").apply(len)
ms_val_matched, assay_val=filter_by_allele_len_pairs(ms_val, assay_val, allele_col="allele", seq_col="seq", filter_both=False)

print("EpiScan R9/RT")
episcan_r9_val_set_for_r9rt=assay_val.query("dataset_source == 'r9'")
episcan_rt_val_set_for_r9rt=assay_val.query("dataset_source == 'rt'")

ms_ap_df=compute_ap_matrix_optimized(ms_val_matched, score_cols, label_col='label', allele_col='allele', disable_tqdm=False)

score_cols=keep_present_cols(episcan_r9_val_set_for_r9rt, score_cols)

n_iterations=100
r9rt_pos_r9rt_neg_per_allele_ap_for_r9rt = mixed_df_per_allele_ap_balanced_fast(episcan_r9_val_set_for_r9rt, episcan_rt_val_set_for_r9rt, score_cols, label_col1='label', label_col2='label', allele_col='allele', neg_ratio=99, n_iterations=n_iterations, seed=42, chunk_size=256, num_threads=16)

assay_ms_val_matched_ms_dfs["episcan_r9rt"]={"ms_val": ms_val, "ms_val_matched": ms_val_matched, "assay_val": assay_val}

assay_ms_val_matched_ms_ap_dfs["episcan_r9rt"]={"ms_val_matched": ms_ap_df, "assay_val": r9rt_pos_r9rt_neg_per_allele_ap_for_r9rt}

EpiScan R9/RT


Iterations: 100%|██████████| 100/100 [00:08<00:00, 12.40it/s]

CPU times: user 1min 30s, sys: 1.38 s, total: 1min 31s
Wall time: 13.8 s


In [ ]:
# get performance for ba con val across steps (for full ms)
assay_val=joint_data["ba_con"]["ba_val"]
ms_val=joint_data["ba_con"]["ms_val"]

assay_alleles=assay_val.allele.unique().tolist()

ms_val["seq_len"]=ms_val.seq.str.replace("$", "").apply(len)
ms_val_matched, assay_val=filter_by_allele_len_pairs(ms_val, assay_val, allele_col="allele", seq_col="seq", filter_both=False)

score_cols=joint_retraining_score_cols("ba_con_retraining_steps_15720_seed_42", save_every_n_steps, 15720, include_final=True, return_map=False)

ms_ap_df=compute_ap_matrix_optimized(ms_val_matched, score_cols, label_col='label', allele_col='allele', disable_tqdm=False)

spearman_df = compute_spearman_matrix_optimized(assay_val, score_cols, reference_col="ba_value", allele_col='allele', disable_tqdm=False)

assay_ms_val_full_ms_dfs["ba_con"]={"ms_val": ms_val, "ms_val_matched": ms_val_matched, "assay_val": assay_val}

assay_ms_val_full_ms_ap_dfs["ba_con"]={"ms_val_matched": ms_ap_df, "assay_val": spearman_df}

100%|██████████| 41/41 [00:04<00:00,  8.45it/s]


In [ ]:
# get performance for ba con val across steps (for matched ms)
assay_val=joint_data_ms_matched["ba_con"]["ba_val"]
ms_val=joint_data_ms_matched["ba_con"]["ms_val"]

assay_alleles=assay_val.allele.unique().tolist()

ms_val["seq_len"]=ms_val.seq.str.replace("$", "").apply(len)
ms_val_matched, assay_val=filter_by_allele_len_pairs(ms_val, assay_val, allele_col="allele", seq_col="seq", filter_both=False)

score_cols=joint_retraining_score_cols("ba_con_retraining_ms_matched_steps_15720_seed_42", save_every_n_steps, 15720, include_final=True, return_map=False)

ms_ap_df=compute_ap_matrix_optimized(ms_val_matched, score_cols, label_col='label', allele_col='allele', disable_tqdm=False)

spearman_df = compute_spearman_matrix_optimized(assay_val, score_cols, reference_col="ba_value", allele_col='allele', disable_tqdm=False)

assay_ms_val_matched_ms_dfs["ba_con"]={"ms_val": ms_val, "ms_val_matched": ms_val_matched, "assay_val": assay_val}

assay_ms_val_matched_ms_ap_dfs["ba_con"]={"ms_val_matched": ms_ap_df, "assay_val": spearman_df}

100%|██████████| 41/41 [00:04<00:00,  8.50it/s]


In [ ]:
# get performance for escape con val across steps (for full ms)
assay_val=joint_data["escape_con"]["escape_val"]
ms_val=joint_data["escape_con"]["ms_val"]

assay_alleles=assay_val.allele.unique().tolist()

ms_val["seq_len"]=ms_val.seq.str.replace("$", "").apply(len)
ms_val_matched, assay_val=filter_by_allele_len_pairs(ms_val, assay_val, allele_col="allele", seq_col="seq", filter_both=False)

score_cols=joint_retraining_score_cols("escape_con_retraining_steps_15720_seed_42", save_every_n_steps, 15720, include_final=True, return_map=False)

ms_ap_df=compute_ap_matrix_optimized(ms_val_matched, score_cols, label_col='label', allele_col='allele', disable_tqdm=False)

spearman_df = compute_spearman_matrix_optimized(assay_val, score_cols, reference_col="escore", allele_col='allele', disable_tqdm=False)

assay_ms_val_full_ms_dfs["escape_con"]={"ms_val": ms_val, "ms_val_matched": ms_val_matched, "assay_val": assay_val}

assay_ms_val_full_ms_ap_dfs["escape_con"]={"ms_val_matched": ms_ap_df, "assay_val": spearman_df}

100%|██████████| 41/41 [00:04<00:00,  8.64it/s]


In [ ]:
# get performance for escape con val across steps (for matched ms)
assay_val=joint_data_ms_matched["escape_con"]["escape_val"]
ms_val=joint_data_ms_matched["escape_con"]["ms_val"]

assay_alleles=assay_val.allele.unique().tolist()

ms_val["seq_len"]=ms_val.seq.str.replace("$", "").apply(len)
ms_val_matched, assay_val=filter_by_allele_len_pairs(ms_val, assay_val, allele_col="allele", seq_col="seq", filter_both=False)

score_cols=joint_retraining_score_cols("escape_con_retraining_ms_matched_steps_15720_seed_42", save_every_n_steps, 15720, include_final=True, return_map=False)

ms_ap_df=compute_ap_matrix_optimized(ms_val_matched, score_cols, label_col='label', allele_col='allele', disable_tqdm=False)

spearman_df = compute_spearman_matrix_optimized(assay_val, score_cols, reference_col="escore", allele_col='allele', disable_tqdm=False)

assay_ms_val_matched_ms_dfs["escape_con"]={"ms_val": ms_val, "ms_val_matched": ms_val_matched, "assay_val": assay_val}

assay_ms_val_matched_ms_ap_dfs["escape_con"]={"ms_val_matched": ms_ap_df, "assay_val": spearman_df}

100%|██████████| 41/41 [00:04<00:00,  8.72it/s]


In [ ]:
# full ms yd
yd_joint_loss_df=build_joint_retraining_loss_trajectory(
    training_stats=f"{DATA_DIR}/Joint_Retraining/yd/training_stats.pkl", # NOTE change here
    assay_val_df=assay_ms_val_full_ms_dfs["yd"]["assay_val"],
    # ms_val_df=assay_ms_val_full_ms_dfs["yd"]["ms_val_matched"],
    ms_val_df=assay_ms_val_full_ms_dfs["yd"]["ms_val"], # NOTE: full ms_val df for loss computation
    base_col="yd_retraining_steps_15720_seed_42",
    checkpoint_steps=save_every_n_steps,
    final_step=15720,
    assay_loss_type="logsmoothap",
    assay_alpha=10.0,
    ms_alpha=10.0,
    assay_batch_size=3072,
    ms_batch_size=3072,
    assay_neg_ratio=1.0,
    ms_neg_ratio=1.0,
    validation_sampling_seed=42000,
    pairwise_chunk_size=2048,
    device="cuda",
)

# matched ms
yd_joint_ms_matched_loss_df=build_joint_retraining_loss_trajectory(
    training_stats=f"{DATA_DIR}/Joint_Retraining_MS_Matched/yd/training_stats.pkl", # NOTE change here
    assay_val_df=assay_ms_val_matched_ms_dfs["yd"]["assay_val"],
    ms_val_df=assay_ms_val_matched_ms_dfs["yd"]["ms_val_matched"],
    base_col="yd_retraining_ms_matched_steps_15720_seed_42",
    checkpoint_steps=save_every_n_steps,
    final_step=15720,
    assay_loss_type="logsmoothap",
    assay_alpha=10.0,
    ms_alpha=10.0,
    assay_batch_size=3072,
    ms_batch_size=3072,
    assay_neg_ratio=1.0,
    ms_neg_ratio=1.0,
    validation_sampling_seed=42000,
    pairwise_chunk_size=2048,
    device="cuda",
)

Loading training stats: /home/pchati/PepCL_Manuscript/Data/Joint_Retraining/yd/training_stats.pkl
Loaded 15,720 training steps.
Validation checkpoints requested: 159

Sampling assay validation batch...
Sampling MS validation batch...

Validation sampling summary:
dataset  n_sampled  n_pos  n_neg  n_alleles
  assay       1446    723    723         16
     ms       3072   1536   1536        142



Validation losses: 100%|██████████| 159/159 [00:00<00:00, 211.98it/s, step=15720]



Validation checkpoints reconstructed: 159
Done.
Loading training stats: /home/pchati/PepCL_Manuscript/Data/Joint_Retraining_MS_Matched/yd/training_stats.pkl
Loaded 15,720 training steps.
Validation checkpoints requested: 159

Sampling assay validation batch...
Sampling MS validation batch...

Validation sampling summary:
dataset  n_sampled  n_pos  n_neg  n_alleles
  assay       1446    723    723         16
     ms       1600    800    800         16



Validation losses: 100%|██████████| 159/159 [00:00<00:00, 354.55it/s, step=15720]



Validation checkpoints reconstructed: 159
Done.


In [ ]:
# full ms episcan r9
episcan_r9_joint_loss_df=build_joint_retraining_loss_trajectory(
    training_stats=f"{DATA_DIR}/Joint_Retraining/episcan_r9/training_stats.pkl", # NOTE change here
    assay_val_df=assay_ms_val_full_ms_dfs["episcan_r9"]["assay_val"],
    # ms_val_df=assay_ms_val_full_ms_dfs["episcan_r9"]["ms_val_matched"],
    ms_val_df=assay_ms_val_full_ms_dfs["episcan_r9"]["ms_val"], # NOTE: full ms_val df for loss computation
    base_col="episcan_r9_retraining_steps_15720_seed_42",
    checkpoint_steps=save_every_n_steps,
    final_step=15720,
    assay_loss_type="logsmoothap",
    assay_alpha=10.0,
    ms_alpha=10.0,
    assay_batch_size=3072,
    ms_batch_size=3072,
    assay_neg_ratio=1.0,
    ms_neg_ratio=1.0,
    validation_sampling_seed=42000,
    pairwise_chunk_size=2048,
    device="cuda",
)

# matched ms episcan r9
episcan_r9_joint_ms_matched_loss_df=build_joint_retraining_loss_trajectory(
    training_stats=f"{DATA_DIR}/Joint_Retraining_MS_Matched/episcan_r9/training_stats.pkl", # NOTE change here
    assay_val_df=assay_ms_val_matched_ms_dfs["episcan_r9"]["assay_val"],
    ms_val_df=assay_ms_val_matched_ms_dfs["episcan_r9"]["ms_val_matched"],
    base_col="episcan_r9_retraining_ms_matched_steps_15720_seed_42",
    checkpoint_steps=save_every_n_steps,
    final_step=15720,
    assay_loss_type="logsmoothap",
    assay_alpha=10.0,
    ms_alpha=10.0,
    assay_batch_size=3072,
    ms_batch_size=3072,
    assay_neg_ratio=1.0,
    ms_neg_ratio=1.0,
    validation_sampling_seed=42000,
    pairwise_chunk_size=2048,
    device="cuda",
)

Loading training stats: /home/pchati/PepCL_Manuscript/Data/Joint_Retraining/episcan_r9/training_stats.pkl
Loaded 15,720 training steps.
Validation checkpoints requested: 159

Sampling assay validation batch...
Sampling MS validation batch...

Validation sampling summary:
dataset  n_sampled  n_pos  n_neg  n_alleles
  assay        676    338    338          4
     ms       3072   1536   1536        142



Validation losses: 100%|██████████| 159/159 [00:00<00:00, 300.14it/s, step=15720]



Validation checkpoints reconstructed: 159
Done.
Loading training stats: /home/pchati/PepCL_Manuscript/Data/Joint_Retraining_MS_Matched/episcan_r9/training_stats.pkl
Loaded 15,720 training steps.
Validation checkpoints requested: 159

Sampling assay validation batch...
Sampling MS validation batch...

Validation sampling summary:
dataset  n_sampled  n_pos  n_neg  n_alleles
  assay        676    338    338          4
     ms        400    200    200          4



Validation losses: 100%|██████████| 159/159 [00:00<00:00, 321.52it/s, step=15720]



Validation checkpoints reconstructed: 159
Done.


In [ ]:
# full ms episcan r9/rt
episcan_r9rt_joint_loss_df=build_joint_retraining_loss_trajectory(
    training_stats=f"{DATA_DIR}/Joint_Retraining/episcan_r9rt/training_stats.pkl", # NOTE change here
    assay_val_df=assay_ms_val_full_ms_dfs["episcan_r9rt"]["assay_val"],
    # ms_val_df=assay_ms_val_full_ms_dfs["episcan_r9rt"]["ms_val_matched"],
    ms_val_df=assay_ms_val_full_ms_dfs["episcan_r9rt"]["ms_val"], # NOTE: full ms_val df for loss computation
    base_col="episcan_r9rt_retraining_steps_15720_seed_42",
    checkpoint_steps=save_every_n_steps,
    final_step=15720,
    assay_loss_type="logsmoothap",
    assay_alpha=10.0,
    ms_alpha=10.0,
    assay_batch_size=3072,
    ms_batch_size=3072,
    assay_neg_ratio=1.0,
    ms_neg_ratio=1.0,
    validation_sampling_seed=42000,
    pairwise_chunk_size=2048,
    device="cuda",
)

# matched ms episcan r9/rt
episcan_r9rt_joint_ms_matched_loss_df=build_joint_retraining_loss_trajectory(
    training_stats=f"{DATA_DIR}/Joint_Retraining_MS_Matched/episcan_r9rt/training_stats.pkl", # NOTE change here
    assay_val_df=assay_ms_val_matched_ms_dfs["episcan_r9rt"]["assay_val"],
    ms_val_df=assay_ms_val_matched_ms_dfs["episcan_r9rt"]["ms_val_matched"],
    base_col="episcan_r9rt_retraining_ms_matched_steps_15720_seed_42",
    checkpoint_steps=save_every_n_steps,
    final_step=15720,
    assay_loss_type="logsmoothap",
    assay_alpha=10.0,
    ms_alpha=10.0,
    assay_batch_size=3072,
    ms_batch_size=3072,
    assay_neg_ratio=1.0,
    ms_neg_ratio=1.0,
    validation_sampling_seed=42000,
    pairwise_chunk_size=2048,
    device="cuda",
)



Loading training stats: /home/pchati/PepCL_Manuscript/Data/Joint_Retraining/episcan_r9rt/training_stats.pkl
Loaded 15,720 training steps.
Validation checkpoints requested: 159

Sampling assay validation batch...


Sampling MS validation batch...

Validation sampling summary:
dataset  n_sampled  n_pos  n_neg  n_alleles
  assay        676    338    338          4
     ms       3072   1536   1536        142



Validation losses: 100%|██████████| 159/159 [00:00<00:00, 441.44it/s, step=15720]



Validation checkpoints reconstructed: 159
Done.
Loading training stats: /home/pchati/PepCL_Manuscript/Data/Joint_Retraining_MS_Matched/episcan_r9rt/training_stats.pkl
Loaded 15,720 training steps.
Validation checkpoints requested: 159

Sampling assay validation batch...
Sampling MS validation batch...

Validation sampling summary:
dataset  n_sampled  n_pos  n_neg  n_alleles
  assay        676    338    338          4
     ms        400    200    200          4



Validation losses: 100%|██████████| 159/159 [00:00<00:00, 470.15it/s, step=15720]



Validation checkpoints reconstructed: 159
Done.


In [ ]:
# full ms ba con
ba_con_joint_loss_df=build_joint_retraining_loss_trajectory(
    training_stats=f"{DATA_DIR}/Joint_Retraining/ba_con/training_stats.pkl", # NOTE change here
    assay_val_df=assay_ms_val_full_ms_dfs["ba_con"]["assay_val"],
    # ms_val_df=assay_ms_val_full_ms_dfs["ba_con"]["ms_val_matched"],
    ms_val_df=assay_ms_val_full_ms_dfs["ba_con"]["ms_val"], # NOTE: full ms_val df for loss computation
    base_col="ba_con_retraining_steps_15720_seed_42",
    checkpoint_steps=save_every_n_steps,
    final_step=15720,
    assay_loss_type="spearman",
    continuous_target_col="ba_value",
    assay_alpha=0.01,
    ms_alpha=10.0,
    assay_batch_size=3072,
    ms_batch_size=3072,
    assay_neg_ratio=1.0,
    ms_neg_ratio=1.0,
    validation_sampling_seed=42000,
    pairwise_chunk_size=2048,
    device="cuda",
)

# matched ms ba con
ba_con_joint_ms_matched_loss_df=build_joint_retraining_loss_trajectory(
    training_stats=f"{DATA_DIR}/Joint_Retraining_MS_Matched/ba_con/training_stats.pkl", # NOTE change here
    assay_val_df=assay_ms_val_matched_ms_dfs["ba_con"]["assay_val"],
    ms_val_df=assay_ms_val_matched_ms_dfs["ba_con"]["ms_val_matched"],
    base_col="ba_con_retraining_ms_matched_steps_15720_seed_42",
    checkpoint_steps=save_every_n_steps,
    final_step=15720,
    assay_loss_type="spearman",
    continuous_target_col="ba_value",
    assay_alpha=0.01,
    ms_alpha=10.0,
    assay_batch_size=3072,
    ms_batch_size=3072,
    assay_neg_ratio=1.0,
    ms_neg_ratio=1.0,
    validation_sampling_seed=42000,
    pairwise_chunk_size=2048,
    device="cuda",
)


Loading training stats: /home/pchati/PepCL_Manuscript/Data/Joint_Retraining/ba_con/training_stats.pkl
Loaded 15,720 training steps.
Validation checkpoints requested: 159

Sampling assay validation batch...
Sampling MS validation batch...

Validation sampling summary:
dataset  n_sampled  n_pos  n_neg  n_alleles
  assay       3052   1526   1526         41
     ms       3072   1536   1536        142



Validation losses: 100%|██████████| 159/159 [00:00<00:00, 368.87it/s, step=15720]



Validation checkpoints reconstructed: 159
Done.
Loading training stats: /home/pchati/PepCL_Manuscript/Data/Joint_Retraining_MS_Matched/ba_con/training_stats.pkl
Loaded 15,720 training steps.
Validation checkpoints requested: 159

Sampling assay validation batch...
Sampling MS validation batch...

Validation sampling summary:
dataset  n_sampled  n_pos  n_neg  n_alleles
  assay       3052   1526   1526         41
     ms       3072   1536   1536         41



Validation losses: 100%|██████████| 159/159 [00:00<00:00, 369.88it/s, step=15720]



Validation checkpoints reconstructed: 159
Done.


In [ ]:
# full ms escape con
escape_con_joint_loss_df=build_joint_retraining_loss_trajectory(
    training_stats=f"{DATA_DIR}/Joint_Retraining/escape_con/training_stats.pkl", # NOTE change here
    assay_val_df=assay_ms_val_full_ms_dfs["escape_con"]["assay_val"],
    # ms_val_df=assay_ms_val_full_ms_dfs["escape_con"]["ms_val_matched"],
    ms_val_df=assay_ms_val_full_ms_dfs["escape_con"]["ms_val"], # NOTE: full ms_val df for loss computation
    base_col="escape_con_retraining_steps_15720_seed_42",
    checkpoint_steps=save_every_n_steps,
    final_step=15720,
    assay_loss_type="spearman",
    continuous_target_col="escore",
    assay_alpha=0.01,
    ms_alpha=10.0,
    assay_batch_size=3072,
    ms_batch_size=3072,
    assay_neg_ratio=1.0,
    ms_neg_ratio=1.0,
    validation_sampling_seed=42000,
    pairwise_chunk_size=2048,
    device="cuda",
)


# matched ms escape con
escape_con_joint_ms_matched_loss_df=build_joint_retraining_loss_trajectory(
    training_stats=f"{DATA_DIR}/Joint_Retraining_MS_Matched/escape_con/training_stats.pkl", # NOTE change here
    assay_val_df=assay_ms_val_matched_ms_dfs["escape_con"]["assay_val"],
    ms_val_df=assay_ms_val_matched_ms_dfs["escape_con"]["ms_val_matched"],
    base_col="escape_con_retraining_ms_matched_steps_15720_seed_42",
    checkpoint_steps=save_every_n_steps,
    final_step=15720,
    assay_loss_type="spearman",
    continuous_target_col="escore",
    assay_alpha=0.01,
    ms_alpha=10.0,
    assay_batch_size=3072,
    ms_batch_size=3072,
    assay_neg_ratio=1.0,
    ms_neg_ratio=1.0,
    validation_sampling_seed=42000,
    pairwise_chunk_size=2048,
    device="cuda",
)

Loading training stats: /home/pchati/PepCL_Manuscript/Data/Joint_Retraining/escape_con/training_stats.pkl
Loaded 15,720 training steps.
Validation checkpoints requested: 159

Sampling assay validation batch...
Sampling MS validation batch...

Validation sampling summary:
dataset  n_sampled  n_pos  n_neg  n_alleles
  assay       1304    657    647         41
     ms       3072   1536   1536        142



Validation losses: 100%|██████████| 159/159 [00:00<00:00, 374.12it/s, step=15720]



Validation checkpoints reconstructed: 159
Done.
Loading training stats: /home/pchati/PepCL_Manuscript/Data/Joint_Retraining_MS_Matched/escape_con/training_stats.pkl
Loaded 15,720 training steps.
Validation checkpoints requested: 159

Sampling assay validation batch...
Sampling MS validation batch...

Validation sampling summary:
dataset  n_sampled  n_pos  n_neg  n_alleles
  assay       1304    657    647         41
     ms       3072   1536   1536         37



Validation losses: 100%|██████████| 159/159 [00:00<00:00, 382.29it/s, step=15720]



Validation checkpoints reconstructed: 159
Done.


In [ ]:
full_ms_cls_joint_plot_dict={
    "YD":{
        "loss_dfs": yd_joint_loss_df,
        "assay_perf": assay_ms_val_full_ms_ap_dfs["yd"]["assay_val"],
        "ms_perf": assay_ms_val_full_ms_ap_dfs["yd"]["ms_val_matched"],
        "base_col":"yd_retraining_steps_15720_seed_42",
        "final_step": 15720,
        "checkpoint_steps": save_every_n_steps,
    },

    "EpiScan R9":{
        "loss_dfs": episcan_r9_joint_loss_df,
        "assay_perf": assay_ms_val_full_ms_ap_dfs["episcan_r9"]["assay_val"],
        "ms_perf": assay_ms_val_full_ms_ap_dfs["episcan_r9"]["ms_val_matched"],
        "base_col":"episcan_r9_retraining_steps_15720_seed_42",
        "final_step": 15720,
        "checkpoint_steps": save_every_n_steps,
    },

    "EpiScan R9/RT":{
        "loss_dfs": episcan_r9rt_joint_loss_df,
        "assay_perf": assay_ms_val_full_ms_ap_dfs["episcan_r9rt"]["assay_val"],
        "ms_perf": assay_ms_val_full_ms_ap_dfs["episcan_r9rt"]["ms_val_matched"],
        "base_col":"episcan_r9rt_retraining_steps_15720_seed_42",
        "final_step": 15720,
        "checkpoint_steps": save_every_n_steps,
    },

}

full_ms_con_joint_plot_dict={
    "BA":{
        "loss_dfs": ba_con_joint_loss_df,
        "assay_perf": assay_ms_val_full_ms_ap_dfs["ba_con"]["assay_val"],
        "ms_perf": assay_ms_val_full_ms_ap_dfs["ba_con"]["ms_val_matched"],
        "base_col":"ba_con_retraining_steps_15720_seed_42",
        "final_step": 15720,
        "checkpoint_steps": save_every_n_steps,
    },

    "ESCAPE-seq":{
        "loss_dfs": escape_con_joint_loss_df,
        "assay_perf": assay_ms_val_full_ms_ap_dfs["escape_con"]["assay_val"],
        "ms_perf": assay_ms_val_full_ms_ap_dfs["escape_con"]["ms_val_matched"],
        "base_col":"escape_con_retraining_steps_15720_seed_42",
        "final_step": 15720,
        "checkpoint_steps": save_every_n_steps,
    },

}

matched_ms_cls_joint_plot_dict={
    "YD":{
        "loss_dfs": yd_joint_ms_matched_loss_df,
        "assay_perf": assay_ms_val_matched_ms_ap_dfs["yd"]["assay_val"],
        "ms_perf": assay_ms_val_matched_ms_ap_dfs["yd"]["ms_val_matched"],
        "base_col":"yd_retraining_ms_matched_steps_15720_seed_42",
        "final_step": 15720,
        "checkpoint_steps": save_every_n_steps,
    },

    "EpiScan R9":{
        "loss_dfs": episcan_r9_joint_ms_matched_loss_df,
        "assay_perf": assay_ms_val_matched_ms_ap_dfs["episcan_r9"]["assay_val"],
        "ms_perf": assay_ms_val_matched_ms_ap_dfs["episcan_r9"]["ms_val_matched"],
        "base_col":"episcan_r9_retraining_ms_matched_steps_15720_seed_42",
        "final_step": 15720,
        "checkpoint_steps": save_every_n_steps,
    },

    "EpiScan R9/RT":{
        "loss_dfs": episcan_r9rt_joint_ms_matched_loss_df,
        "assay_perf": assay_ms_val_matched_ms_ap_dfs["episcan_r9rt"]["assay_val"],
        "ms_perf": assay_ms_val_matched_ms_ap_dfs["episcan_r9rt"]["ms_val_matched"],
        "base_col":"episcan_r9rt_retraining_ms_matched_steps_15720_seed_42",
        "final_step": 15720,
        "checkpoint_steps": save_every_n_steps,
    },

}

matched_ms_con_joint_plot_dict={
    "BA":{
        "loss_dfs": ba_con_joint_ms_matched_loss_df,
        "assay_perf": assay_ms_val_matched_ms_ap_dfs["ba_con"]["assay_val"],
        "ms_perf": assay_ms_val_matched_ms_ap_dfs["ba_con"]["ms_val_matched"],
        "base_col":"ba_con_retraining_ms_matched_steps_15720_seed_42",
        "final_step": 15720,
        "checkpoint_steps": save_every_n_steps,
    },

    "ESCAPE-seq":{
        "loss_dfs": escape_con_joint_ms_matched_loss_df,
        "assay_perf": assay_ms_val_matched_ms_ap_dfs["escape_con"]["assay_val"],
        "ms_perf": assay_ms_val_matched_ms_ap_dfs["escape_con"]["ms_val_matched"],
        "base_col":"escape_con_retraining_ms_matched_steps_15720_seed_42",
        "final_step": 15720,
        "checkpoint_steps": save_every_n_steps,
    },

}


joint_updating_results_dict={
    "full_ms_cls_joint_plot_dict": full_ms_cls_joint_plot_dict,
    "full_ms_con_joint_plot_dict": full_ms_con_joint_plot_dict,
    "matched_ms_cls_joint_plot_dict": matched_ms_cls_joint_plot_dict,
    "matched_ms_con_joint_plot_dict": matched_ms_con_joint_plot_dict,
}

# save_dict_pickle(joint_updating_results_dict, f"{DATA_DIR}/Revisions_Data/joint_updating_results_loss_plotting.pkl")

In [ ]:
# get matched ms and assay points and pareto optimal
yd_es_points=find_joint_retraining_descriptive_points(
    assay_perf=assay_ms_val_full_ms_ap_dfs["yd"]["assay_val"],
    ms_perf=assay_ms_val_full_ms_ap_dfs["yd"]["ms_val_matched"],
    base_col="yd_retraining_steps_15720_seed_42",
    checkpoint_steps=save_every_n_steps,
    final_step=15720,
    joint_method="minmax_sum",
)


yd_es_points_ms_matched=find_joint_retraining_descriptive_points(
    assay_perf=assay_ms_val_matched_ms_ap_dfs["yd"]["assay_val"],
    ms_perf=assay_ms_val_matched_ms_ap_dfs["yd"]["ms_val_matched"],
    base_col="yd_retraining_ms_matched_steps_15720_seed_42",
    checkpoint_steps=save_every_n_steps,
    final_step=15720,
    joint_method="minmax_sum",
)

print("Full MS points:", yd_es_points["early_stopping_steps"])
print("MS matched points:", yd_es_points_ms_matched["early_stopping_steps"])

Full MS points: [3701, 8101, 15101]
MS matched points: [1801, 1901, 4901]


In [ ]:
# episcan r9
episcan_r9_es_points=find_joint_retraining_descriptive_points(
    assay_perf=assay_ms_val_full_ms_ap_dfs["episcan_r9"]["assay_val"],
    ms_perf=assay_ms_val_full_ms_ap_dfs["episcan_r9"]["ms_val_matched"],
    base_col="episcan_r9_retraining_steps_15720_seed_42",
    checkpoint_steps=save_every_n_steps,
    final_step=15720,
    joint_method="minmax_sum",
)


episcan_r9_es_points_ms_matched=find_joint_retraining_descriptive_points(
    assay_perf=assay_ms_val_matched_ms_ap_dfs["episcan_r9"]["assay_val"],
    ms_perf=assay_ms_val_matched_ms_ap_dfs["episcan_r9"]["ms_val_matched"],
    base_col="episcan_r9_retraining_ms_matched_steps_15720_seed_42",
    checkpoint_steps=save_every_n_steps,
    final_step=15720,
   joint_method="minmax_sum",
)

print("Full MS points:", episcan_r9_es_points["early_stopping_steps"])
print("MS matched points:", episcan_r9_es_points_ms_matched["early_stopping_steps"])

Full MS points: [8501, 13201, 14201]
MS matched points: [2301, 3501, 12501]


In [ ]:
# episcan r9/rt
episcan_r9rt_es_points=find_joint_retraining_descriptive_points(
    assay_perf=assay_ms_val_full_ms_ap_dfs["episcan_r9rt"]["assay_val"],
    ms_perf=assay_ms_val_full_ms_ap_dfs["episcan_r9rt"]["ms_val_matched"],
    base_col="episcan_r9rt_retraining_steps_15720_seed_42",
    checkpoint_steps=save_every_n_steps,
    final_step=15720,
    joint_method="minmax_sum",
)


episcan_r9rt_es_points_ms_matched=find_joint_retraining_descriptive_points(
    assay_perf=assay_ms_val_matched_ms_ap_dfs["episcan_r9rt"]["assay_val"],
    ms_perf=assay_ms_val_matched_ms_ap_dfs["episcan_r9rt"]["ms_val_matched"],
    base_col="episcan_r9rt_retraining_ms_matched_steps_15720_seed_42",
    checkpoint_steps=save_every_n_steps,
    final_step=15720,
    joint_method="minmax_sum",

)

print("Full MS points:", episcan_r9rt_es_points["early_stopping_steps"])
print("MS matched points:", episcan_r9rt_es_points_ms_matched["early_stopping_steps"])

Full MS points: [6601, 10901]
MS matched points: [1901, 3501]


In [ ]:
# ba con
ba_con_es_points=find_joint_retraining_descriptive_points(
    assay_perf=assay_ms_val_full_ms_ap_dfs["ba_con"]["assay_val"],
    ms_perf=assay_ms_val_full_ms_ap_dfs["ba_con"]["ms_val_matched"],
    base_col="ba_con_retraining_steps_15720_seed_42",
    checkpoint_steps=save_every_n_steps,
    final_step=15720,
    joint_method="minmax_sum",

)

ba_con_es_points_ms_matched=find_joint_retraining_descriptive_points(
    assay_perf=assay_ms_val_matched_ms_ap_dfs["ba_con"]["assay_val"],
    ms_perf=assay_ms_val_matched_ms_ap_dfs["ba_con"]["ms_val_matched"],
    base_col="ba_con_retraining_ms_matched_steps_15720_seed_42",
    checkpoint_steps=save_every_n_steps,
    final_step=15720,
    joint_method="minmax_sum",

)

print("Full MS points:", ba_con_es_points["early_stopping_steps"])
print("MS matched points:", ba_con_es_points_ms_matched["early_stopping_steps"])

Full MS points: [4601, 12701, 15701]
MS matched points: [6401, 14701, 15101]


In [ ]:
# escape con
escape_con_es_points=find_joint_retraining_descriptive_points(
    assay_perf=assay_ms_val_full_ms_ap_dfs["escape_con"]["assay_val"],
    ms_perf=assay_ms_val_full_ms_ap_dfs["escape_con"]["ms_val_matched"],
    base_col="escape_con_retraining_steps_15720_seed_42",
    checkpoint_steps=save_every_n_steps,
    final_step=15720,
    joint_method="minmax_sum",
)


escape_con_es_points_ms_matched=find_joint_retraining_descriptive_points(
    assay_perf=assay_ms_val_matched_ms_ap_dfs["escape_con"]["assay_val"],
    ms_perf=assay_ms_val_matched_ms_ap_dfs["escape_con"]["ms_val_matched"],
    base_col="escape_con_retraining_ms_matched_steps_15720_seed_42",
    checkpoint_steps=save_every_n_steps,
    final_step=15720,
    joint_method="minmax_sum",
)

print("Full MS points:", escape_con_es_points["early_stopping_steps"])
print("MS matched points:", escape_con_es_points_ms_matched["early_stopping_steps"])


Full MS points: [6301, 13501]
MS matched points: [1901, 4601]


In [ ]:
# evaluate selected points per assay

# init dicts (full ms)
assay_ms_test_full_ms_dfs={}
assay_ms_test_full_ms_ap_dfs={}

# init dicts (matched ms)
assay_ms_test_matched_ms_dfs={}
assay_ms_test_matched_ms_ap_dfs={}

# NOTE: USE 5 SEED VERSION FOR COMPARING TO FT AND PEPCL

In [ ]:
%%time
# get performance for holec val across steps (for full ms)
score_cols=joint_retraining_score_cols("yd_retraining_steps_15720_seed_42", save_every_n_steps, 15720, include_final=True, return_map=False)

assay_test=joint_data_es['yd']['holec_test']
assay_alleles=assay_test.allele.unique().tolist()
ms_test=joint_data_es['yd']['ms_test']

ms_test["seq_len"]=ms_test.seq.str.replace("$", "").apply(len)
ms_test_matched, assay_test=filter_by_allele_len_pairs(ms_test, assay_test, allele_col="allele", seq_col="seq", filter_both=False)

ms_ap_df=compute_ap_matrix_optimized(ms_test_matched, score_cols, label_col='label', allele_col='allele', disable_tqdm=False)

assay_ap_df=compute_ap_matrix_optimized(assay_test, score_cols, label_col='label', allele_col='allele', disable_tqdm=False)

assay_ms_test_full_ms_dfs["yd"]={"ms_test": ms_test, "ms_test_matched": ms_test_matched, "assay_test": assay_test}

assay_ms_test_full_ms_ap_dfs["yd"]={"ms_test_matched": ms_ap_df, "assay_test": assay_ap_df}

100%|██████████| 16/16 [00:00<00:00, 143.21it/s]

CPU times: user 5.47 s, sys: 1.04 s, total: 6.51 s
Wall time: 6.5 s


In [ ]:
%%time
# get performance for holec val across steps (for matched ms)
score_cols=joint_retraining_score_cols("yd_retraining_ms_matched_steps_15720_seed_42", save_every_n_steps, 15720, include_final=True, return_map=False)

assay_test=joint_data_ms_matched_es['yd']['holec_test']
assay_alleles=assay_test.allele.unique().tolist()
ms_test=joint_data_ms_matched_es['yd']['ms_test']

ms_test["seq_len"]=ms_test.seq.str.replace("$", "").apply(len)
ms_test_matched, assay_test=filter_by_allele_len_pairs(ms_test, assay_test, allele_col="allele", seq_col="seq", filter_both=False)

ms_ap_df=compute_ap_matrix_optimized(ms_test_matched, score_cols, label_col='label', allele_col='allele', disable_tqdm=False)

assay_ap_df=compute_ap_matrix_optimized(assay_test, score_cols, label_col='label', allele_col='allele', disable_tqdm=False)

assay_ms_test_matched_ms_dfs["yd"]={"ms_test": ms_test, "ms_test_matched": ms_test_matched, "assay_test": assay_test}

assay_ms_test_matched_ms_ap_dfs["yd"]={"ms_test_matched": ms_ap_df, "assay_test": assay_ap_df}

100%|██████████| 16/16 [00:00<00:00, 114.99it/s]

CPU times: user 5.85 s, sys: 787 ms, total: 6.64 s
Wall time: 6.62 s


In [ ]:
%%time
# get performance for episcan r9 val across steps (for full ms)

score_cols=joint_retraining_score_cols("episcan_r9_retraining_steps_15720_seed_42", save_every_n_steps, 15720, include_final=True, return_map=False)

assay_test=joint_data_es["episcan_r9"]["episcan_r9_rt_test"]
ms_test=joint_data_es["episcan_r9"]["ms_test"]

ms_test["seq_len"]=ms_test.seq.str.replace("$", "").apply(len)
ms_test_matched, assay_test=filter_by_allele_len_pairs(ms_test, assay_test, allele_col="allele", seq_col="seq", filter_both=False)

ms_ap_df=compute_ap_matrix_optimized(ms_test_matched, score_cols, label_col='label', allele_col='allele', disable_tqdm=False)

n_iterations=100
allele_pos_counts = {"A0201": 100, "A0301": 100, "B0801": 100, "B5701": 100}

# per-allele AP matrix, pos-focused (negs from r9 only)
print("EpiScan R9")
episcan_r9_test_set_for_r9=assay_test.query("dataset_source == 'r9'")
episcan_rt_test_set_for_r9=assay_test.query("dataset_source == 'rt'")

score_cols=keep_present_cols(episcan_r9_test_set_for_r9, score_cols)

r9rt_pos_r9_neg_per_allele_ap_for_r9 = mixed_df_per_allele_ap_pos_focused_fast(episcan_r9_test_set_for_r9, episcan_rt_test_set_for_r9.query("label == 1"), score_cols, allele_pos_counts, label_col1='label', label_col2='label', allele_col='allele', neg_ratio=99, n_iterations=n_iterations, seed=42, chunk_size=256, num_threads=16)

assay_test_mixture=pd.concat([episcan_r9_test_set_for_r9, episcan_rt_test_set_for_r9.query("label == 1")])

assay_ms_test_full_ms_dfs["episcan_r9"]={"ms_test": ms_test, "ms_test_matched": ms_test_matched, "assay_test": assay_test_mixture} # use proper assay val mixture

assay_ms_test_full_ms_ap_dfs["episcan_r9"]={"ms_test_matched": ms_ap_df, "assay_test": r9rt_pos_r9_neg_per_allele_ap_for_r9}

100%|██████████| 4/4 [00:00<00:00, 30.70it/s]


EpiScan R9


Iterations: 100%|██████████| 100/100 [00:01<00:00, 63.92it/s]

CPU times: user 8.46 s, sys: 944 ms, total: 9.4 s
Wall time: 5.69 s


In [ ]:
%%time
# get performance for episcan r9 val across steps (for matched ms)
score_cols=joint_retraining_score_cols("episcan_r9_retraining_ms_matched_steps_15720_seed_42", save_every_n_steps, 15720, include_final=True, return_map=False)

assay_test=joint_data_ms_matched_es["episcan_r9"]["episcan_r9_rt_test"]
ms_test=joint_data_ms_matched_es["episcan_r9"]["ms_test"]

ms_test["seq_len"]=ms_test.seq.str.replace("$", "").apply(len)
ms_test_matched, assay_test=filter_by_allele_len_pairs(ms_test, assay_test, allele_col="allele", seq_col="seq", filter_both=False)

ms_ap_df=compute_ap_matrix_optimized(ms_test_matched, score_cols, label_col='label', allele_col='allele', disable_tqdm=False)

n_iterations=100
allele_pos_counts = {"A0201": 100, "A0301": 100, "B0801": 100, "B5701": 100}

# per-allele AP matrix, pos-focused (negs from r9 only)
print("EpiScan R9")
episcan_r9_test_set_for_r9=assay_test.query("dataset_source == 'r9'")
episcan_rt_test_set_for_r9=assay_test.query("dataset_source == 'rt'")

score_cols=keep_present_cols(episcan_r9_test_set_for_r9, score_cols)

r9rt_pos_r9_neg_per_allele_ap_for_r9 = mixed_df_per_allele_ap_pos_focused_fast(episcan_r9_test_set_for_r9, episcan_rt_test_set_for_r9.query("label == 1"), score_cols, allele_pos_counts, label_col1='label', label_col2='label', allele_col='allele', neg_ratio=99, n_iterations=n_iterations, seed=42, chunk_size=256, num_threads=16)

assay_test_mixture=pd.concat([episcan_r9_test_set_for_r9, episcan_rt_test_set_for_r9.query("label == 1")])

assay_ms_test_matched_ms_dfs["episcan_r9"]={"ms_test": ms_test, "ms_test_matched": ms_test_matched, "assay_test": assay_test_mixture} # use proper assay val mixture

assay_ms_test_matched_ms_ap_dfs["episcan_r9"]={"ms_test_matched": ms_ap_df, "assay_test": r9rt_pos_r9_neg_per_allele_ap_for_r9}

100%|██████████| 4/4 [00:00<00:00, 21.31it/s]


EpiScan R9


Iterations: 100%|██████████| 100/100 [00:01<00:00, 71.10it/s]

CPU times: user 10.5 s, sys: 1.52 s, total: 12 s
Wall time: 8.35 s


In [ ]:
%%time
# get performance for episcan r9/rt val across steps (for full ms)
score_cols=joint_retraining_score_cols("episcan_r9rt_retraining_steps_15720_seed_42", save_every_n_steps, 15720, include_final=True, return_map=False)

assay_test=joint_data_es["episcan_r9rt"]["episcan_r9_rt_test"]
ms_test=joint_data_es["episcan_r9rt"]["ms_test"]

assay_alleles=assay_test.allele.unique().tolist()

ms_test["seq_len"]=ms_test.seq.str.replace("$", "").apply(len)
ms_test_matched, assay_test=filter_by_allele_len_pairs(ms_test, assay_test, allele_col="allele", seq_col="seq", filter_both=False)

print("EpiScan R9/RT")
episcan_r9_test_set_for_r9rt=assay_test.query("dataset_source == 'r9'")
episcan_rt_test_set_for_r9rt=assay_test.query("dataset_source == 'rt'")

score_cols=keep_present_cols(episcan_r9_test_set_for_r9rt, score_cols)

ms_ap_df=compute_ap_matrix_optimized(ms_test_matched, score_cols, label_col='label', allele_col='allele', disable_tqdm=False)

n_iterations=100
allele_pos_counts = {"A0201": 100, "A0301": 100, "B0801": 100, "B5701": 100}

r9rt_pos_r9rt_neg_per_allele_ap_for_r9rt = mixed_df_per_allele_ap_both_fast(episcan_r9_test_set_for_r9rt, episcan_rt_test_set_for_r9rt, score_cols, allele_pos_counts, label_col1='label', label_col2='label', allele_col='allele', neg_ratio=99, n_iterations=n_iterations, seed=42, chunk_size=256, num_threads=16)


assay_ms_test_full_ms_dfs["episcan_r9rt"]={"ms_test": ms_test, "ms_test_matched": ms_test_matched, "assay_test": assay_test}

assay_ms_test_full_ms_ap_dfs["episcan_r9rt"]={"ms_test_matched": ms_ap_df, "assay_test": r9rt_pos_r9rt_neg_per_allele_ap_for_r9rt}

EpiScan R9/RT


Iterations: 100%|██████████| 100/100 [00:01<00:00, 54.54it/s]

CPU times: user 10.7 s, sys: 1.32 s, total: 12 s
Wall time: 7.68 s


In [ ]:
%%time
# get performance for episcan r9/rt val across steps (for matched ms)
score_cols=joint_retraining_score_cols("episcan_r9rt_retraining_ms_matched_steps_15720_seed_42", save_every_n_steps, 15720, include_final=True, return_map=False)

assay_test=joint_data_ms_matched_es["episcan_r9rt"]["episcan_r9_rt_test"]
ms_test=joint_data_ms_matched_es["episcan_r9rt"]["ms_test"]

assay_alleles=assay_test.allele.unique().tolist()

ms_test["seq_len"]=ms_test.seq.str.replace("$", "").apply(len)
ms_test_matched, assay_test=filter_by_allele_len_pairs(ms_test, assay_test, allele_col="allele", seq_col="seq", filter_both=False)

print("EpiScan R9/RT")
episcan_r9_test_set_for_r9rt=assay_test.query("dataset_source == 'r9'")
episcan_rt_test_set_for_r9rt=assay_test.query("dataset_source == 'rt'")

score_cols=keep_present_cols(episcan_r9_test_set_for_r9rt, score_cols)

ms_ap_df=compute_ap_matrix_optimized(ms_test_matched, score_cols, label_col='label', allele_col='allele', disable_tqdm=False)

n_iterations=100
allele_pos_counts = {"A0201": 100, "A0301": 100, "B0801": 100, "B5701": 100}

r9rt_pos_r9rt_neg_per_allele_ap_for_r9rt = mixed_df_per_allele_ap_both_fast(episcan_r9_test_set_for_r9rt, episcan_rt_test_set_for_r9rt, score_cols, allele_pos_counts, label_col1='label', label_col2='label', allele_col='allele', neg_ratio=99, n_iterations=n_iterations, seed=42, chunk_size=256, num_threads=16)


assay_ms_test_matched_ms_dfs["episcan_r9rt"]={"ms_test": ms_test, "ms_test_matched": ms_test_matched, "assay_test": assay_test}

assay_ms_test_matched_ms_ap_dfs["episcan_r9rt"]={"ms_test_matched": ms_ap_df, "assay_test": r9rt_pos_r9rt_neg_per_allele_ap_for_r9rt}

EpiScan R9/RT


Iterations: 100%|██████████| 100/100 [00:02<00:00, 46.59it/s]

CPU times: user 11.3 s, sys: 1.45 s, total: 12.8 s
Wall time: 8.97 s


In [ ]:
# get performance for ba con val across steps (for full ms)
assay_test=joint_data_es["ba_con"]["ba_test"]
ms_test=joint_data_es["ba_con"]["ms_test"]

assay_alleles=assay_test.allele.unique().tolist()

ms_test["seq_len"]=ms_test.seq.str.replace("$", "").apply(len)
ms_test_matched, assay_test=filter_by_allele_len_pairs(ms_test, assay_test, allele_col="allele", seq_col="seq", filter_both=False)

score_cols=joint_retraining_score_cols("ba_con_retraining_steps_15720_seed_42", save_every_n_steps, 15720, include_final=True, return_map=False)

ms_ap_df=compute_ap_matrix_optimized(ms_test_matched, score_cols, label_col='label', allele_col='allele', disable_tqdm=False)

spearman_df = compute_spearman_matrix_optimized(assay_test, score_cols, reference_col="ba_value", allele_col='allele', disable_tqdm=False)

assay_ms_test_full_ms_dfs["ba_con"]={"ms_test": ms_test, "ms_test_matched": ms_test_matched, "assay_test": assay_test}

assay_ms_test_full_ms_ap_dfs["ba_con"]={"ms_test_matched": ms_ap_df, "assay_test": spearman_df}

100%|██████████| 51/51 [00:00<00:00, 265.40it/s]


In [ ]:
# get performance for ba con val across steps (for matched ms)
assay_test=joint_data_ms_matched_es["ba_con"]["ba_test"]
ms_test=joint_data_ms_matched_es["ba_con"]["ms_test"]

assay_alleles=assay_test.allele.unique().tolist()

ms_test["seq_len"]=ms_test.seq.str.replace("$", "").apply(len)
ms_test_matched, assay_test=filter_by_allele_len_pairs(ms_test, assay_test, allele_col="allele", seq_col="seq", filter_both=False)

score_cols=joint_retraining_score_cols("ba_con_retraining_ms_matched_steps_15720_seed_42", save_every_n_steps, 15720, include_final=True, return_map=False)

ms_ap_df=compute_ap_matrix_optimized(ms_test_matched, score_cols, label_col='label', allele_col='allele', disable_tqdm=False)

spearman_df = compute_spearman_matrix_optimized(assay_test, score_cols, reference_col="ba_value", allele_col='allele', disable_tqdm=False)

assay_ms_test_matched_ms_dfs["ba_con"]={"ms_test": ms_test, "ms_test_matched": ms_test_matched, "assay_test": assay_test}

assay_ms_test_matched_ms_ap_dfs["ba_con"]={"ms_test_matched": ms_ap_df, "assay_test": spearman_df}

100%|██████████| 51/51 [00:00<00:00, 273.94it/s]


In [ ]:
# get performance for escape con val across steps (for full ms)
assay_test=joint_data_es["escape_con"]["escape_test"]
ms_test=joint_data_es["escape_con"]["ms_test"]

assay_alleles=assay_test.allele.unique().tolist()

ms_test["seq_len"]=ms_test.seq.str.replace("$", "").apply(len)
ms_test_matched, assay_test=filter_by_allele_len_pairs(ms_test, assay_test, allele_col="allele", seq_col="seq", filter_both=False)

score_cols=joint_retraining_score_cols("escape_con_retraining_steps_15720_seed_42", save_every_n_steps, 15720, include_final=True, return_map=False)

ms_ap_df=compute_ap_matrix_optimized(ms_test_matched, score_cols, label_col='label', allele_col='allele', disable_tqdm=False)

spearman_df = compute_spearman_matrix_optimized(assay_test, score_cols, reference_col="escore", allele_col='allele', disable_tqdm=False)

assay_ms_test_full_ms_dfs["escape_con"]={"ms_test": ms_test, "ms_test_matched": ms_test_matched, "assay_test": assay_test}

assay_ms_test_full_ms_ap_dfs["escape_con"]={"ms_test_matched": ms_ap_df, "assay_test": spearman_df}

100%|██████████| 41/41 [00:00<00:00, 337.85it/s]


In [ ]:
# get performance for escape con val across steps (for matched ms)
assay_test=joint_data_ms_matched_es["escape_con"]["escape_test"]
ms_test=joint_data_ms_matched_es["escape_con"]["ms_test"]

assay_alleles=assay_test.allele.unique().tolist()

ms_test["seq_len"]=ms_test.seq.str.replace("$", "").apply(len)
ms_test_matched, assay_test=filter_by_allele_len_pairs(ms_test, assay_test, allele_col="allele", seq_col="seq", filter_both=False)

score_cols=joint_retraining_score_cols("escape_con_retraining_ms_matched_steps_15720_seed_42", save_every_n_steps, 15720, include_final=True, return_map=False)

ms_ap_df=compute_ap_matrix_optimized(ms_test_matched, score_cols, label_col='label', allele_col='allele', disable_tqdm=False)

spearman_df = compute_spearman_matrix_optimized(assay_test, score_cols, reference_col="escore", allele_col='allele', disable_tqdm=False)

assay_ms_test_matched_ms_dfs["escape_con"]={"ms_test": ms_test, "ms_test_matched": ms_test_matched, "assay_test": assay_test}

assay_ms_test_matched_ms_ap_dfs["escape_con"]={"ms_test_matched": ms_ap_df, "assay_test": spearman_df}

100%|██████████| 41/41 [00:00<00:00, 145.40it/s]


In [ ]:
# set all inputs
# set dicts for summary outputs
# assay_ms_val_full_ms_ap_dfs
# assay_ms_val_matched_ms_ap_dfs

# assay_ms_test_full_ms_ap_dfs
# assay_ms_test_matched_ms_ap_dfs

summary_dfs_val={
    "yd": holec_summary_df_val,
    "episcan_r9": episcan_r9_summary_df_val,
    "episcan_r9rt": episcan_r9rt_summary_df_val,
    "ba_con": ba_con_summary_df_val,
    "escape_con": escape_con_summary_df_val,
}

summary_dfs_test={
    "yd": holec_summary_df_test,
    "episcan_r9": episcan_r9_summary_df_test,
    "episcan_r9rt": episcan_r9rt_summary_df_test,
    "ba_con": ba_con_summary_df_test,
    "escape_con": escape_con_summary_df_test,
}

full_ms_es_selected_columns_per_assay={
    "yd": yd_es_points["selected_columns"],
    "episcan_r9": episcan_r9_es_points["selected_columns"],
    "episcan_r9rt": episcan_r9rt_es_points["selected_columns"],
    "ba_con": ba_con_es_points["selected_columns"],
    "escape_con": escape_con_es_points["selected_columns"],
}

matched_ms_es_selected_columns_per_assay={
    "yd": yd_es_points_ms_matched["selected_columns"],
    "episcan_r9": episcan_r9_es_points_ms_matched["selected_columns"],
    "episcan_r9rt": episcan_r9rt_es_points_ms_matched["selected_columns"],
    "ba_con": ba_con_es_points_ms_matched["selected_columns"],
    "escape_con": escape_con_es_points_ms_matched["selected_columns"],
}

dataset_order=['yd', 'episcan_r9', 'episcan_r9rt', 'ba_con', 'escape_con']
dataset_labels={
    'yd': 'YD',
    'episcan_r9': 'EpiScan R9',
    'episcan_r9rt': 'EpiScan R9/RT',
    'ba_con': 'BA',
    'escape_con': 'ESCAPE-seq'
}

In [ ]:
retraining_selected_val_test_summary=summarize_retraining_val_test_selected_points(
    assay_ms_val_full_ms_ap_dfs=assay_ms_val_full_ms_ap_dfs,
    assay_ms_val_matched_ms_ap_dfs=assay_ms_val_matched_ms_ap_dfs,
    assay_ms_test_full_ms_ap_dfs=assay_ms_test_full_ms_ap_dfs,
    assay_ms_test_matched_ms_ap_dfs=assay_ms_test_matched_ms_ap_dfs,

    full_ms_es_selected_columns_per_assay=full_ms_es_selected_columns_per_assay,
    matched_ms_es_selected_columns_per_assay=matched_ms_es_selected_columns_per_assay,

    dataset_order=dataset_order,
    dataset_labels=dataset_labels,
    summary_dfs_val=summary_dfs_val,
    summary_dfs_test=summary_dfs_test,
    config_names=["MHCPrime", "FT", "Replay only", "Full PepCL",],
    config_labels={"MHCPrime":"MHCPrime", "FT":"FT", "Replay only": "Replay only", "Full PepCL":"PepCL",},
    verbose=False,

)

In [ ]:
# save for plotting
joint_updating_point_selection_results=dict(    
    assay_ms_val_full_ms_ap_dfs=assay_ms_val_full_ms_ap_dfs,
    assay_ms_val_matched_ms_ap_dfs=assay_ms_val_matched_ms_ap_dfs,
    assay_ms_test_full_ms_ap_dfs=assay_ms_test_full_ms_ap_dfs,
    assay_ms_test_matched_ms_ap_dfs=assay_ms_test_matched_ms_ap_dfs,
    full_ms_es_selected_columns_per_assay=full_ms_es_selected_columns_per_assay,
    matched_ms_es_selected_columns_per_assay=matched_ms_es_selected_columns_per_assay,
    dataset_order=dataset_order,
    dataset_labels=dataset_labels,
)

# save_dict_pickle(joint_updating_point_selection_results, f"{DATA_DIR}/Revisions_Data/joint_updating_point_selection_results.pkl")

In [ ]:
# save for plotting
joint_updating_cls_plotting=dict(
    assay_ms_val_full_ms_ap_dfs=assay_ms_val_full_ms_ap_dfs,
    assay_ms_val_matched_ms_ap_dfs=assay_ms_val_matched_ms_ap_dfs,
    assay_ms_test_full_ms_ap_dfs=assay_ms_test_full_ms_ap_dfs,
    assay_ms_test_matched_ms_ap_dfs=assay_ms_test_matched_ms_ap_dfs,
    full_ms_es_selected_columns_per_assay=full_ms_es_selected_columns_per_assay,
    matched_ms_es_selected_columns_per_assay=matched_ms_es_selected_columns_per_assay,
    summary_dfs_val=summary_dfs_val,
    summary_dfs_test=summary_dfs_test,
)

joint_updating_con_plotting=dict(
    assay_ms_val_full_ms_ap_dfs=assay_ms_val_full_ms_ap_dfs,
    assay_ms_val_matched_ms_ap_dfs=assay_ms_val_matched_ms_ap_dfs,
    assay_ms_test_full_ms_ap_dfs=assay_ms_test_full_ms_ap_dfs,
    assay_ms_test_matched_ms_ap_dfs=assay_ms_test_matched_ms_ap_dfs,
    full_ms_es_selected_columns_per_assay=full_ms_es_selected_columns_per_assay,
    matched_ms_es_selected_columns_per_assay=matched_ms_es_selected_columns_per_assay,
    summary_dfs_val=summary_dfs_val,
    summary_dfs_test=summary_dfs_test,
)

# save_dict_pickle(joint_updating_cls_plotting, f"{DATA_DIR}/Revisions_Data/joint_updating_cls_plotting.pkl")

# save_dict_pickle(joint_updating_con_plotting, f"{DATA_DIR}/Revisions_Data/joint_updating_con_plotting.pkl")